# Notebook 03 — Flood Extent Validation + Stakeholder Dashboard

This notebook validates the **modelled flood extent** against **GFM satellite-observed flood extent**
for the Cagayan River Basin, Philippines, and produces an interactive HTML dashboard for stakeholders.

**Pipeline**: River discharge → Return period (EVT/POT) → JRC depth maps (CLIMADA-Petals) → Modelled extent → **Validation metrics**

**Outputs**:
- **CSV metrics**: Confusion matrix (TP/FP/FN/TN), F1, IoU, Precision, Recall, Bias at multiple depth thresholds
- **Population exposure tables**: Affected population by municipality (modelled vs. observed)
- **Interactive HTML dashboard**: Leaflet map + Plotly charts, suitable for non-technical stakeholders

**Design principle**: Validates binary flood extent (presence/absence), not depth accuracy. Population figures are an **impact proxy** only.

**Prerequisites**:
- ✅ Notebook 1 calibration complete (EVT parameters)
- ✅ Notebook 2 hazard integration complete (flood depth maps)
- ✅ GFM observed extent data available (GeoTIFF files with dates in filenames)

**Estimated runtime**: ~30–60 minutes per flood episode (first run; cached thereafter)

**Documentation**: See [Notebook 3 Validation Guide](../../docs/user-guides/notebook03-validation-guide.md) for a detailed walkthrough.


## 0) Environment & imports


In [ ]:
from __future__ import annotations

import os
import re
import json
import math
import base64
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

import geopandas as gpd
import xarray as xr

from shapely.geometry import box

import rasterio
from rasterio.merge import merge as rio_merge
from rasterio.mask import mask as rio_mask
from rasterio.warp import reproject, Resampling
from rasterio.enums import Resampling as RioResampling

import matplotlib.pyplot as plt

# Notebook display helper
try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)
    print("IPythom not available, install it")


# Optional: faster zonal stats if available
try:
    from rasterstats import zonal_stats
    RASTERSTATS_AVAILABLE = True
except Exception:
    RASTERSTATS_AVAILABLE = False
    print("rasterstats not available, install it")

# --- CLIMADA / PETALS ---
CLIMADA_AVAILABLE = True
try:
    from climada_petals.hazard.rf_glofas.transform_ops import (
        regrid as petals_regrid,
        flood_depth as petals_flood_depth,
        apply_flopros as petals_apply_flopros,
    )
    from climada_petals.hazard.rf_glofas.setup import download_flopros_database
except Exception as e:
    CLIMADA_AVAILABLE = False
    print("❌ CLIMADA-Petals import failed:", e)
    print("   This notebook requires CLIMADA-Petals for regrid + flood_depth + FLOPROS masking.")
    print("   Fix your environment, then restart the kernel.")

# --- Project helper (EVT POT discharge -> RP) ---
try:
    from philflood.calibration.evt_pot import discharge_to_return_period_pot  # preferred if running inside repo env
except Exception:
    # fallback: import local file if notebook is run outside installed package
    try:
        from philflood.calibration.evt_pot import discharge_to_return_period_pot
    except Exception as e:
        raise ImportError("Could not import discharge_to_return_period_pot from philflood.calibration.evt_pot or evt_pot.py") from e

print("✅ Imports complete.")


## 0.5) Pre-flight check (fast fail + actionable diagnostics)

Run this cell once. If it fails, fix before continuing.

**Checks performed**:
- ✅ All required Python packages installed (numpy, pandas, geopandas, rasterio, xarray, CLIMADA-Petals)
- ✅ Data files exist (HYBAS, ADM3, WorldPop, GFM directory)
- ✅ Output directory is writable

**Troubleshooting**: See [Notebook 03 Issues](../../docs/user-guides/troubleshooting.md#notebook-03-validation-issues)


## 1) Robust paths (repo-root detection + defaults)


In [ ]:
REPO_ROOT_MANUAL = None  # e.g., Path(r"C:\pipelines\GLOFAS_ImpactFloodForecasting_PHL")

def find_repo_root(start: Optional[Path] = None) -> Path:
    markers = ["pyproject.toml", "setup.cfg", "setup.py", ".git", "src"]
    p = (start or Path.cwd()).resolve()
    if p.is_file():
        p = p.parent
    for parent in (p, *p.parents):
        if any((parent / m).exists() for m in markers):
            return parent
    raise RuntimeError("Could not find repo root automatically. Set REPO_ROOT_MANUAL.")

if REPO_ROOT_MANUAL is None:
    try:
        REPO_ROOT = find_repo_root()
    except Exception:
        REPO_ROOT = Path(r"C:\pipelines\GLOFAS_ImpactFloodForecasting_PHL")
else:
    REPO_ROOT = Path(REPO_ROOT_MANUAL)

REPO_ROOT = REPO_ROOT.resolve()
print("REPO_ROOT =", REPO_ROOT)

DATA_RAW = REPO_ROOT / "data" / "raw"
DATA_PROCESSED = REPO_ROOT / "data" / "processed"
DATA_INTERIM = REPO_ROOT / "data" / "interim"

# --- Key input paths (all relative to REPO_ROOT, no hardcoded absolute paths) ---
# HydroBASINS Level 7 — Africa bundle (has all sidecar files)
HYBAS_L7_SHP = DATA_RAW / "vectors" / "hydrobasins" / "africa" / "hybas_af_lev01-12_v1c" / "hybas_af_lev07_v1c.shp"
# Admin Level 3 — Liberia GADM v4.1 clan boundaries (305 clans)
ADM3_GEOJSON = DATA_RAW / "vectors" / "admin" / "lbr_cod_ab" / "gadm41_LBR_3.json"
# WorldPop 100m population raster — Liberia 2025
WORLDPOP_RASTER = DATA_RAW / "worldpop" / "LBR" / "lbr_pop_2025_CN_100m_R2025A_v1.tif"
# GFM satellite-observed flood extent — place event GeoTIFFs here when available
GFM_VALIDATION_ROOT = DATA_INTERIM / "validation" / "GFM" / "saint_paul_01"

JRC_RAW_ROOT = DATA_RAW / "jrc_flood_maps"

# Output root for this notebook
VALIDATION_OUT_ROOT = DATA_PROCESSED / "validation"
OUT_DIR = VALIDATION_OUT_ROOT  # Main output directory for validation results
print("Defaults set.")

In [ ]:
import sys, platform
import importlib.metadata as im

def _pkg_version(name: str) -> str:
    try:
        return im.version(name)
    except Exception:
        return "not-installed"

print("🧪 Pre-flight check")
print("Python:", sys.version.split()[0], "|", platform.platform())
print("Repo root:", REPO_ROOT)

required = [
    ("numpy","numpy"),
    ("pandas","pandas"),
    ("geopandas","geopandas"),
    ("rasterio","rasterio"),
    ("xarray","xarray"),
    ("pyarrow","pyarrow"),
    ("requests","requests"),
    ("Pillow","Pillow"),
]
print("\n📦 Packages:")
for label, pkg in required:
    print(f"  - {label:10s}: {_pkg_version(pkg)}")

if not CLIMADA_AVAILABLE:
    raise RuntimeError("CLIMADA-Petals is required but not importable. Activate the correct environment and restart the kernel.")

def _assert_exists(p: Path, label: str):
    if not Path(p).exists():
        raise FileNotFoundError(f"Missing {label}: {p}")
    print(f"  ✅ {label}: {p}")

print("\n📁 Required inputs:")
_assert_exists(HYBAS_L7_SHP, "HYBAS_L7_SHP")
_assert_exists(ADM3_GEOJSON, "ADM3_GEOJSON")
_assert_exists(WORLDPOP_RASTER, "WORLDPOP_RASTER")
_assert_exists(GFM_VALIDATION_ROOT, "GFM_VALIDATION_ROOT")
_assert_exists(JRC_RAW_ROOT, "JRC_RAW_ROOT")

OUT_DIR.mkdir(parents=True, exist_ok=True)
tmp = OUT_DIR / "_preflight_write_test.txt"
tmp.write_text("ok", encoding="utf-8")
tmp.unlink(missing_ok=True)
print("\n✅ Output directory writable:", OUT_DIR)

print("\n✅ Pre-flight passed (basic).")

## 2) Auto-detect calibration run (like Notebook 02)

Automatically finds and loads your latest Notebook 1 calibration output.

**Loaded**:
- Basin ID or municipality selection
- EVT parameters (parquet file path)
- Discharge timeseries directory
- Selection mode (basin vs. municipality)

**To manually specify**: Set `AUTO_DETECT = False` and provide `BASIN_ID_HINT` and `RUN_TAG_HINT`.

In [ ]:
AUTO_DETECT = True
BASIN_ID_HINT = None   # e.g. "Cagayan_01"
RUN_TAG_HINT = None    # e.g. "2026-01-19_calib-test"

def find_latest_run_config(processed_root: Path, basin_id_hint: Optional[str]=None, run_tag_hint: Optional[str]=None) -> Path:
    calib_root = processed_root / "calibration" / "evt_pot"
    if basin_id_hint and run_tag_hint:
        p = calib_root / basin_id_hint / run_tag_hint / "run_config.json"
        if not p.exists():
            raise FileNotFoundError(f"run_config.json not found at {p}")
        return p

    candidates = list(calib_root.glob("*/*/run_config.json"))
    if basin_id_hint:
        candidates = [c for c in candidates if c.parent.parent.name == basin_id_hint]
    if not candidates:
        raise FileNotFoundError(f"No run_config.json found under: {calib_root}")
    candidates.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return candidates[0]

if AUTO_DETECT:
    run_config_path = find_latest_run_config(DATA_PROCESSED, BASIN_ID_HINT, RUN_TAG_HINT)
    run_config = json.loads(run_config_path.read_text(encoding="utf-8"))
    selection_mode = run_config.get("selection_mode", "basin")
    USE_MUNI_AOI = bool(run_config.get("use_muni_aoi", selection_mode == "municipality"))
    BASIN_ID = run_config.get("basin_id", "MUNI_SELECTION")
    RUN_TAG = run_config.get("run_tag")
    
    # Retrocompatibility: support both old (adm3_names) and new (adm3_ids) formats
    if "adm3_ids" in run_config:
        SELECTED_MUNIS = run_config["adm3_ids"]  # New format: list of IDs
    elif "selected_municipalities" in run_config:
        # Old format: could be list of names or count
        sm = run_config["selected_municipalities"]
        SELECTED_MUNIS = sm if isinstance(sm, list) else []
    else:
        SELECTED_MUNIS = []
    
    print("✅ Auto-detected run config:", run_config_path)
    print(json.dumps(run_config, indent=2))
else:
    USE_MUNI_AOI = True
    BASIN_ID = BASIN_ID_HINT or "MUNI_SELECTION"
    RUN_TAG = RUN_TAG_HINT or "2026-01-19_calib-test"
    SELECTED_MUNIS = []
    print("⚠️ Manual configuration in use.")


EVT_PARAMS_PARQUET = DATA_PROCESSED / "calibration" / "evt_pot" / BASIN_ID / RUN_TAG / "results" / "evt_pot_calibration.parquet"
TIMESERIES_DIR = DATA_PROCESSED / "calibration" / "evt_pot" / BASIN_ID / RUN_TAG / "timeseries"

OUT_DIR = VALIDATION_OUT_ROOT / BASIN_ID / RUN_TAG
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("EVT_PARAMS_PARQUET =", EVT_PARAMS_PARQUET)
print("TIMESERIES_DIR     =", TIMESERIES_DIR)
print("OUT_DIR            =", OUT_DIR)

## 3) User controls

Configure validation parameters before running:

- **DECLUSTER_GAP_DAYS**: Group flood dates within N days as the same episode (default: 5)
- **DEPTH_THRESHOLDS_FOR_METRICS**: Depth thresholds for confusion-matrix metrics (default: 0.05–1.5 m)
- **DASHBOARD_THRESHOLD_MIN/MAX/STEP**: Range for the depth slider in the dashboard
- **BASIN_DISPLAY_NAME**: Human-readable basin name shown in the dashboard header
- **EVENT_LABELS**: *(Optional)* Custom display names for flood events in the dashboard dropdown.  
  Auto-generated labels are used as fallback if a key is missing.

In [ ]:
# -----------------------
# USER CONTROLS
# -----------------------

DECLUSTER_GAP_DAYS = 5

# Discharge window rule:
#   window_start = episode_start - DISCHARGE_PAD_DAYS_BEFORE_EPISODE_START days
#   window_end   = episode_end
DISCHARGE_PAD_DAYS_BEFORE_EPISODE_START = 3

# Thresholds used for *metrics* only (fast)
DEPTH_THRESHOLDS_FOR_METRICS = [
    0.00, 0.01, 0.02, 0.03, 0.04, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45,
    0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95,
    1.00, 1.05, 1.10, 1.15, 1.20, 1.25, 1.30, 1.35, 1.40, 1.45,
    1.50, 2.00, 2.50, 3.00
]

# Thresholds for dashboard slider
DASHBOARD_THRESHOLD_MIN  = 0.0
DASHBOARD_THRESHOLD_MAX  = 3.0
DASHBOARD_THRESHOLD_STEP = 0.01

# ── Dashboard display settings ──────────────────────────────────────────────

# Basin name displayed in the dashboard header
BASIN_DISPLAY_NAME = "Cagayan River Basin"

# Custom event labels for the dashboard dropdown.
# Keys   = auto-generated episode IDs (e.g. "EP01_20231214_20231219").
# Values = the label shown to stakeholders (e.g. "Typhoon Egay – Jun 2023").
# If an episode has no entry here, an auto-generated label is used instead.
EVENT_LABELS = {
    # Uncomment and fill in as needed — keep the keys as the auto-detected EP IDs:
    # "EP01_20231214_20231219": "Typhoon Egay – Jun 2023",
    # "EP02_20240101_20240107": "Northeast Monsoon – Jan 2024",
}

# ── Pipeline settings ────────────────────────────────────────────────────────

# Compute FLOPROS (flood protection) scenario in the pipeline.
# NOTE: The stakeholder dashboard always shows the "No Protection" view;
# FLOPROS is computed for pipeline completeness and is not exposed in the UI.
ENABLE_FLOPROS_SCENARIOS = True

# Dashboard output file
DASHBOARD_FILENAME = OUT_DIR / "validation_dashboard.html"

# Internal dummy tests (set True only during development)
RUN_INTERNAL_DUMMY_TESTS = False

print("USE_MUNI_AOI  =", USE_MUNI_AOI)
print("BASIN_DISPLAY_NAME =", BASIN_DISPLAY_NAME)
print("SELECTED_MUNIS =", SELECTED_MUNIS[:10], "..." if len(SELECTED_MUNIS) > 10 else "")
print("EVENT_LABELS   =", EVENT_LABELS if EVENT_LABELS else "(none — auto labels will be used)")

## 4) Build AOI (basin vs municipality) + ADM3 aggregation layer

Constructs Area of Interest (AOI) boundary:

**Municipality mode**: Union of selected ADM3 polygons (by `adm3_id`)  
**Basin mode**: Union of HydroBASINS L7 polygons intersecting GFM extent (auto-detected)

**Output**: `AOI_BOUNDARY` geometry (EPSG:4326, WGS84 lat/lon)

**Check**: Verify polygon count and total area match expectations.

In [ ]:

def load_adm3(adm3_path: Path) -> gpd.GeoDataFrame:
    gdf = gpd.read_file(adm3_path)
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326")
    else:
        gdf = gdf.to_crs("EPSG:4326")
    assert "adm3_name" in gdf.columns, "adm3_name missing from ADM3 layer"
    assert "adm3_id" in gdf.columns, "adm3_id missing from ADM3 layer"
    return gdf

adm3_gdf = load_adm3(ADM3_GEOJSON)
print("ADM3 loaded:", len(adm3_gdf))

def gfm_bounds_union(root: Path) -> Optional[Tuple[float,float,float,float]]:
    """Fast scan of all GFM .tif files to get union bounds (west, south, east, north)."""
    tifs = list(Path(root).rglob("*.tif"))
    if not tifs:
        return None
    bounds = None
    for p in tifs:
        try:
            with rasterio.open(p) as src:
                b = src.bounds  # left, bottom, right, top
        except Exception:
            continue
        if bounds is None:
            bounds = [b.left, b.bottom, b.right, b.top]
        else:
            bounds[0] = min(bounds[0], b.left)
            bounds[1] = min(bounds[1], b.bottom)
            bounds[2] = max(bounds[2], b.right)
            bounds[3] = max(bounds[3], b.top)
    return tuple(bounds) if bounds else None

GFM_BOUNDS = gfm_bounds_union(GFM_VALIDATION_ROOT)
if GFM_BOUNDS is None:
    print("⚠️ No GFM tif files found yet; basin-mode AOI auto-selection may fail.")
else:
    print("GFM bounds union:", GFM_BOUNDS)

# Optional manual override for basin selection (only if needed)
HYBAS_ID_FIELD = None          # e.g. "HYBAS_ID"
HYBAS_ID_VALUES = None         # e.g. [1234567890]
BASIN_BBOX_BUFFER_DEG = 0.10   # expands bbox selection slightly

def build_aoi_boundary(use_muni: bool, selected_munis: List[str], adm3_gdf: gpd.GeoDataFrame):
    """
    AOI boundary (EPSG:4326) selection rule:

    - Municipality mode: union of selected ADM3 polygons (by adm3_id).
    - Basin mode: union of HydroBASINS L7 polygons intersecting the GFM union bbox (buffered).
      (This avoids accidentally selecting the entire Australasia L7 layer.)
    """
    if use_muni:
        if not selected_munis:
            raise ValueError("Municipality mode requires SELECTED_MUNIS (adm3_id list).")
        
        # Primary selection: by adm3_id (new format)
        sel = adm3_gdf[adm3_gdf["adm3_id"].isin(selected_munis)].copy()
        
        # Retrocompatibility: if empty, try by adm3_name (old format)
        if sel.empty:
            sel = adm3_gdf[adm3_gdf["adm3_name"].isin(selected_munis)].copy()
            if not sel.empty:
                print("⚠️ Selected municipalities by adm3_name (legacy mode). Consider updating to adm3_id.")
        
        if sel.empty:
            raise ValueError(f"No ADM3 matched SELECTED_MUNIS: {selected_munis}. Check adm3_id values.")
        
        print(f"✅ Municipality mode: Selected {len(sel)} municipality/municipalities by adm3_id")
        return sel.unary_union, "municipality"

    hy = gpd.read_file(HYBAS_L7_SHP)
    hy = hy.to_crs("EPSG:4326") if hy.crs else hy.set_crs("EPSG:4326")

    if HYBAS_ID_FIELD and HYBAS_ID_VALUES is not None:
        if HYBAS_ID_FIELD not in hy.columns:
            raise ValueError(f"HYBAS_ID_FIELD='{HYBAS_ID_FIELD}' not found. Available: {list(hy.columns)}")
        hy_sel = hy[hy[HYBAS_ID_FIELD].isin(HYBAS_ID_VALUES)].copy()
        if hy_sel.empty:
            raise ValueError("HYBAS_ID_VALUES did not match any basin polygons.")
        return hy_sel.unary_union, "basin"

    if GFM_BOUNDS is None:
        raise ValueError(
            "Basin mode requires either HYBAS_ID_FIELD/HYBAS_ID_VALUES, or at least one GFM .tif present "
            "to auto-select basins by intersection."
        )

    west, south, east, north = GFM_BOUNDS
    bbox_geom = box(west - BASIN_BBOX_BUFFER_DEG,
                    south - BASIN_BBOX_BUFFER_DEG,
                    east + BASIN_BBOX_BUFFER_DEG,
                    north + BASIN_BBOX_BUFFER_DEG)

    hy_sel = hy[hy.intersects(bbox_geom)].copy()
    if hy_sel.empty:
        raise ValueError("No HYBAS L7 polygons intersect the GFM bbox. Check AOI and inputs.")
    print(f"✅ Basin-mode AOI selected {len(hy_sel)} HYBAS L7 polygons using GFM bbox (buffer={BASIN_BBOX_BUFFER_DEG}°).")
    return hy_sel.unary_union, "basin"

AOI_BOUNDARY, MODE = build_aoi_boundary(USE_MUNI_AOI, SELECTED_MUNIS, adm3_gdf)
print("MODE =", MODE)


## 5) Observed hazard: GFM daily mosaics → declustered episodes → max extent

Processes GloFAS Flood Maps (observed extent) into validation-ready format.

**Steps**:
1. **Date parsing**: Reads GFM filenames, extracts dates (YYYYMMDD or YYYY-MM-DD)
2. **Declustering**: Groups dates into flood episodes (5-day gap rule by default)
3. **Daily mosaicking**: Merges multiple GFM tiles per date
4. **Episode maximum**: Computes per-episode maximum extent ("where did water reach at any point?")

**Output**: `obs_extent__EPXX_*.tif` files (one per episode)

**Data source**: See [GFM Data Guide](../../docs/getting-started/gfm-data-guide.md) for how to obtain and prepare GFM files.

In [ ]:

def parse_gfm_date_from_filename(path: Path) -> Optional[pd.Timestamp]:
    """Extract date from filename - tries multiple patterns."""
    # Pattern 1: YYYYMMDD anywhere in filename
    m = re.search(r"_(\d{8})(?:\D|$)", path.stem)
    if m:
        return pd.to_datetime(m.group(1), format="%Y%m%d")
    
    # Pattern 2: YYYY-MM-DD
    m = re.search(r"(\d{4}-\d{2}-\d{2})", path.stem)
    if m:
        try:
            return pd.to_datetime(m.group(1))
        except:
            pass
    
    return None

def list_gfm_files(root: Path) -> pd.DataFrame:
    """List all raster files with dates extracted from filenames."""
    tifs = list(Path(root).rglob("*.tif"))
    rows = []
    for p in tifs:
        dt = parse_gfm_date_from_filename(p)
        if dt is None:
            continue
        rows.append({"path": p, "date": dt, "folder": p.parent.name})
    df = pd.DataFrame(rows).sort_values("date")
    return df

gfm_df = list_gfm_files(GFM_VALIDATION_ROOT)
print("GFM tif files:", len(gfm_df))
display(gfm_df.head(10))

def get_nodata_value(src: rasterio.DatasetReader) -> float:
    """Dynamically detect nodata value from raster."""
    if src.nodata is not None:
        return float(src.nodata)
    
    # Common nodata values to check
    common_nodata = [255, 0, -9999, -999, 9999, np.nan]
    
    # Sample first band
    arr = src.read(1)
    
    # Check if any common nodata value is present
    for val in common_nodata:
        if np.isnan(val):
            if np.any(np.isnan(arr)):
                return np.nan
        elif val in arr:
            return float(val)
    
    # Default fallback
    return 255.0

def mosaic_and_clip(paths: List[Path], aoi_geom, nodata_val: int = 255) -> Tuple[np.ndarray, rasterio.Affine, dict]:
    """Mosaic and clip rasters to AOI geometry."""
    srcs = [rasterio.open(str(p)) for p in paths]
    mosaic, out_trans = rio_merge(srcs)
    meta = srcs[0].meta.copy()
    for src in srcs:
        src.close()

    arr = mosaic[0]
    meta.update({"height": arr.shape[0], "width": arr.shape[1], "transform": out_trans})

    with rasterio.io.MemoryFile() as memfile:
        with memfile.open(**meta) as ds:
            ds.write(arr, 1)
            out_image, out_transform = rio_mask(ds, [aoi_geom], crop=True, nodata=nodata_val)
            out_arr = out_image[0]
            out_meta = ds.meta.copy()
            out_meta.update({"height": out_arr.shape[0], "width": out_arr.shape[1], "transform": out_transform})
    return out_arr, out_transform, out_meta

OBS_DAILY_DIR = OUT_DIR / "observed" / "daily"
OBS_EPISODE_DIR = OUT_DIR / "observed" / "episodes"
OBS_DAILY_DIR.mkdir(parents=True, exist_ok=True)
OBS_EPISODE_DIR.mkdir(parents=True, exist_ok=True)

aoi_geom = AOI_BOUNDARY

daily_products = []
for dt, grp in gfm_df.groupby("date"):
    out_path = OBS_DAILY_DIR / f"obs_extent__{dt.strftime('%Y%m%d')}__{MODE}.tif"
    if out_path.exists():
        daily_products.append({"date": dt, "path": out_path})
        continue
    arr, transform, meta = mosaic_and_clip(grp["path"].tolist(), aoi_geom, nodata_val=255)
    meta.update({"dtype": rasterio.uint8, "count": 1, "nodata": 255})
    with rasterio.open(out_path, "w", **meta) as dst:
        dst.write(arr.astype(np.uint8), 1)
    daily_products.append({"date": dt, "path": out_path})

daily_df = pd.DataFrame(daily_products).sort_values("date")
print("Daily mosaics ready:", len(daily_df))
display(daily_df.head())

def decluster_dates(dates: List[pd.Timestamp], gap_days: int) -> List[Dict]:
    """Cluster dates into episodes based on gap threshold."""
    if not dates:
        return []
    dates = sorted(dates)
    episodes = []
    start = dates[0]
    curr = [dates[0]]
    for d in dates[1:]:
        if (d - curr[-1]).days > gap_days:
            episodes.append({"start": start, "end": curr[-1], "dates": curr})
            start = d
            curr = [d]
        else:
            curr.append(d)
    episodes.append({"start": start, "end": curr[-1], "dates": curr})
    return episodes

episodes = decluster_dates(daily_df["date"].tolist(), DECLUSTER_GAP_DAYS)
print("Episodes:", len(episodes))
for i, ep in enumerate(episodes, 1):
    print(f"  EP{i:02d}: {ep['start'].date()} → {ep['end'].date()}  ({len(ep['dates'])} days)")

def resample_to_common_grid(arrays_with_meta: List[Tuple[np.ndarray, rasterio.Affine]]) -> Tuple[List[np.ndarray], rasterio.Affine]:
    """
    Resample all arrays to a common grid (using the first as reference).
    Returns resampled arrays and the reference transform.
    """
    if not arrays_with_meta:
        return [], None
    
    # Use first array as reference
    ref_arr, ref_transform = arrays_with_meta[0]
    ref_shape = ref_arr.shape
    
    resampled = [ref_arr]
    
    # Resample remaining arrays to match reference
    for arr, transform in arrays_with_meta[1:]:
        if arr.shape == ref_shape and transform == ref_transform:
            # Already same grid
            resampled.append(arr)
        else:
            # Need to resample
            dst_array = np.empty(ref_shape, dtype=arr.dtype)
            reproject(
                source=arr,
                destination=dst_array,
                src_transform=transform,
                src_crs="EPSG:4326",
                dst_transform=ref_transform,
                dst_crs="EPSG:4326",
                resampling=Resampling.nearest
            )
            resampled.append(dst_array)
    
    return resampled, ref_transform

print(f"\n🎯 Processing {len(episodes)} flood episodes...")

episode_products = []
for i, ep in enumerate(episodes, 1):
    ep_id = f"EP{i:02d}_{ep['start'].strftime('%Y%m%d')}_{ep['end'].strftime('%Y%m%d')}"
    out_path = OBS_EPISODE_DIR / f"obs_extent__{ep_id}__max.tif"
    
    if out_path.exists():
        print(f"  ✅ {ep_id}: Using cached file")
        episode_products.append({"episode_id": ep_id, "start": ep["start"], "end": ep["end"], "path": out_path})
        continue

    arrays_with_meta = []
    meta0 = None
    valid_data_count = 0
    
    for d in ep["dates"]:
        p = daily_df.loc[daily_df["date"] == d, "path"].iloc[0]
        with rasterio.open(p) as src:
            a = src.read(1).astype("float32")
            nod = get_nodata_value(src)
            
            # Replace nodata with NaN
            if np.isnan(nod):
                pass  # already NaN
            else:
                a = np.where(a == nod, np.nan, a)
            
            # Count valid pixels
            n_valid = np.sum(np.isfinite(a))
            if n_valid > 0:
                valid_data_count += 1
                arrays_with_meta.append((a, src.transform))
            else:
                print(f"    ⚠️ {d.date()}: No valid data (all nodata/NaN), skipping")
            
            if meta0 is None:
                meta0 = src.meta.copy()
    
    # Check if we have any valid data
    if not arrays_with_meta:
        print(f"  ⚠️ {ep_id}: No valid data in any raster - creating empty placeholder")
        # Create empty output
        if meta0 is not None:
            empty_arr = np.full((meta0['height'], meta0['width']), 255, dtype=np.uint8)
            meta0.update({"dtype": rasterio.uint8, "count": 1, "nodata": 255})
            with rasterio.open(out_path, "w", **meta0) as dst:
                dst.write(empty_arr, 1)
        episode_products.append({"episode_id": ep_id, "start": ep["start"], "end": ep["end"], "path": out_path})
        continue
    
    print(f"  📊 {ep_id}: Processing {len(arrays_with_meta)} rasters with valid data (skipped {len(ep['dates']) - len(arrays_with_meta)})")

    # Resample all arrays to common grid
    resampled_arrays, final_transform = resample_to_common_grid(arrays_with_meta)
    
    # Stack and compute max
    stack = np.stack(resampled_arrays, axis=0)
    
    # Suppress all-NaN warning and handle it explicitly
    with np.errstate(invalid='ignore'):
        max_extent = np.nanmax(stack, axis=0)
    
    # Check if result has any valid data
    n_valid_out = np.sum(np.isfinite(max_extent))
    if n_valid_out == 0:
        print(f"  ⚠️ {ep_id}: Max operation resulted in all-NaN (possible complete nodata overlap)")
    else:
        print(f"  ✅ {ep_id}: {n_valid_out:,} valid pixels in max extent")
    
    out = np.where(np.isfinite(max_extent), max_extent, 255).astype(np.uint8)
    
    # Update metadata with final transform
    meta0.update({
        "dtype": rasterio.uint8,
        "count": 1,
        "nodata": 255,
        "height": out.shape[0],
        "width": out.shape[1],
        "transform": final_transform
    })
    
    with rasterio.open(out_path, "w", **meta0) as dst:
        dst.write(out, 1)

    episode_products.append({"episode_id": ep_id, "start": ep["start"], "end": ep["end"], "path": out_path})

episode_df = pd.DataFrame(episode_products).sort_values("start")
print(f"\n✅ Episode max extents ready: {len(episode_df)}")
display(episode_df)


## 6) Discharge → Return Period for each episode

Converts calibrated discharge to return period maps for each flood episode.

**Steps**:
1. **Gauge filtering**: Selects gauges within AOI using L12 cell mapping
2. **Timeseries loading**: Loads discharge for each episode's date range
3. **RP computation**: Applies EVT/POT formula per gauge per day (`discharge_to_return_period_pot`)
4. **Spatial maps**: Creates two RP maps per episode:
   - **Envelope**: Maximum RP over entire episode
   - **Peakday**: RP on day with maximum median discharge

**Output**: `rp__EPXX__envelope.nc` and `rp__EPXX__peakday.nc` (NetCDF format)

**Note**: Prints % of gauges with available data. >70% good, <50% may indicate missing data issues.

In [ ]:
if not EVT_PARAMS_PARQUET.exists():
    raise FileNotFoundError(f"EVT params not found: {EVT_PARAMS_PARQUET}")

evt_params = pd.read_parquet(EVT_PARAMS_PARQUET)
req_cols = {"virtual_gauge_id","threshold_m3s","lambda_events_per_year","gpd_xi","gpd_sigma"}
missing = req_cols - set(evt_params.columns)
if missing:
    raise ValueError(f"EVT params parquet missing columns: {missing}")

evt_params = evt_params.copy()
evt_params["virtual_gauge_id"] = evt_params["virtual_gauge_id"].astype(str)
print("EVT gauges:", len(evt_params))

def parse_lat_lon_from_gauge_id(gid: str) -> Tuple[float,float]:
    mlat = re.search(r"lat_(-?\d+(?:\.\d+)?)", gid)
    mlon = re.search(r"lon_(-?\d+(?:\.\d+)?)", gid)
    if not (mlat and mlon):
        raise ValueError(f"Could not parse lat/lon from virtual_gauge_id: {gid}")
    return float(mlat.group(1)), float(mlon.group(1))

evt_params["lat"] = evt_params["virtual_gauge_id"].apply(lambda s: parse_lat_lon_from_gauge_id(s)[0])
evt_params["lon"] = evt_params["virtual_gauge_id"].apply(lambda s: parse_lat_lon_from_gauge_id(s)[1])

def _pick_lat_lon_columns(df: pd.DataFrame, label: str) -> Tuple[str, str]:
    candidates = [
        ("lat", "lon"),
        ("latitude", "longitude"),
        ("cell_lat", "cell_lon"),
        ("lat_deg", "lon_deg"),
        ("y", "x"),
    ]
    for lat_col, lon_col in candidates:
        if lat_col in df.columns and lon_col in df.columns:
            return lat_col, lon_col
    raise ValueError(f"{label} missing lat/lon columns. Found: {list(df.columns)}")

def _grid_spacing(values: pd.Series) -> float:
    vals = np.sort(values.unique())
    if len(vals) < 2:
        return np.nan
    diffs = np.diff(vals)
    diffs = diffs[diffs > 0]
    return float(np.median(diffs)) if len(diffs) else np.nan

def _build_cell_polygons(df: pd.DataFrame, lat_col: str, lon_col: str) -> Tuple[gpd.GeoSeries, float, float]:
    dx = _grid_spacing(df[lon_col])
    dy = _grid_spacing(df[lat_col])
    if not np.isfinite(dx) or not np.isfinite(dy):
        raise ValueError("Could not infer grid resolution from l12_cell_coordinates")
    half_dx = dx / 2.0
    half_dy = dy / 2.0
    geoms = [
        box(lon - half_dx, lat - half_dy, lon + half_dx, lat + half_dy)
        for lat, lon in zip(df[lat_col], df[lon_col])
    ]
    return gpd.GeoSeries(geoms, crs="EPSG:4326"), dx, dy

# Filter gauges using L12 selection (Notebook 1 logic: L12s intersect AOI -> all gauges in those L12s)
mapping_dir = EVT_PARAMS_PARQUET.parent.parent / "mapping"
l12_vg_path = mapping_dir / "l12_virtual_gauge.parquet"
l12_cells_path = mapping_dir / "l12_cell_coordinates.parquet"
use_aoi_point_filter = False

l12_cells_gdf = None
l12_cells_sel_gdf = None
l12_ids = set()
l12_grid_dx = None
l12_grid_dy = None

if l12_vg_path.exists() and l12_cells_path.exists():
    l12_vg = pd.read_parquet(l12_vg_path)
    l12_cells = pd.read_parquet(l12_cells_path)
    req_vg = {"l12_id", "virtual_gauge_id"}
    missing_vg = req_vg - set(l12_vg.columns)
    if missing_vg:
        raise ValueError(f"l12_virtual_gauge.parquet missing columns: {missing_vg}")
    if "l12_id" not in l12_cells.columns:
        raise ValueError(f"l12_cell_coordinates.parquet missing l12_id column: {l12_cells_path}")
    lat_col, lon_col = _pick_lat_lon_columns(l12_cells, "l12_cell_coordinates.parquet")
    l12_cells = l12_cells[["l12_id", lat_col, lon_col]].copy()
    l12_cells["l12_id"] = l12_cells["l12_id"].astype(str)
    l12_polys, l12_grid_dx, l12_grid_dy = _build_cell_polygons(l12_cells, lat_col, lon_col)
    l12_cells_gdf = gpd.GeoDataFrame(l12_cells, geometry=l12_polys, crs="EPSG:4326")
    
    # Use cell polygons to include cells that barely touch the AOI
    l12_cells_sel_gdf = l12_cells_gdf[l12_cells_gdf.intersects(AOI_BOUNDARY)].copy()
    l12_ids = set(l12_cells_sel_gdf["l12_id"])
    if not l12_ids:
        raise ValueError("No L12 cells intersect the AOI boundary. Check AOI selection and mapping files.")
    l12_vg["l12_id"] = l12_vg["l12_id"].astype(str)
    allowed_gauges = set(l12_vg[l12_vg["l12_id"].isin(l12_ids)]["virtual_gauge_id"].astype(str))
    before = len(evt_params)
    evt_params = evt_params[evt_params["virtual_gauge_id"].isin(allowed_gauges)].copy()
    print(f"EVT gauges after L12->AOI filter: {len(evt_params)} / {before} (L12s: {len(l12_ids)})")
elif l12_vg_path.exists():
    l12_vg = pd.read_parquet(l12_vg_path)
    if "virtual_gauge_id" not in l12_vg.columns:
        raise ValueError(f"l12_virtual_gauge.parquet missing virtual_gauge_id column: {l12_vg_path}")
    allowed_gauges = set(l12_vg["virtual_gauge_id"].astype(str))
    before = len(evt_params)
    evt_params = evt_params[evt_params["virtual_gauge_id"].isin(allowed_gauges)].copy()
    print(f"EVT gauges after L12 mapping filter (no AOI L12 selection): {len(evt_params)} / {before}")
    print("⚠️ L12 cell coordinates not found; using all gauges in mapping file.")
else:
    print(f"⚠️ L12 mapping files not found, falling back to AOI boundary filter: {l12_vg_path}")
    use_aoi_point_filter = True

# Build points for diagnostics or fallback filtering
evt_points = gpd.GeoDataFrame(
    evt_params.copy(),
    geometry=gpd.points_from_xy(evt_params["lon"], evt_params["lat"]),
    crs="EPSG:4326"
 )
if use_aoi_point_filter:
    evt_points = evt_points[evt_points.intersects(AOI_BOUNDARY)].copy()
    if len(evt_points) == 0:
        raise ValueError("No EVT virtual gauges intersect the AOI boundary. Check basin_id/run_tag and AOI selection.")
    print(f"EVT gauges in AOI (point filter): {len(evt_points)} / {len(evt_params)}")
    evt_params = evt_points.drop(columns="geometry")
else:
    aoi_count = int(evt_points.intersects(AOI_BOUNDARY).sum())
    print(f"EVT gauges intersect AOI (point check only): {aoi_count} / {len(evt_points)}")
    evt_params = evt_points.drop(columns="geometry")

# FIX: Remove duplicate VG__ gauges (cell-level CELL__ gauges have timeseries)
if evt_params["virtual_gauge_id"].str.startswith("VG__").any():
    vg_count = evt_params["virtual_gauge_id"].str.startswith("VG__").sum()
    print(f"⚠️ Found {vg_count} VG__ (pour point) gauges mixed with CELL__ gauges")
    print(f"   Filtering to keep only CELL__ gauges (cell-level extraction)")
    evt_params = evt_params[evt_params["virtual_gauge_id"].str.startswith("CELL__")].copy()
    evt_points = evt_points[evt_points["virtual_gauge_id"].str.startswith("CELL__")].copy()
    print(f"   After filtering: {len(evt_params)} gauges (removed {vg_count} VG__ duplicates)")

if not TIMESERIES_DIR.exists():
    raise FileNotFoundError(f"TIMESERIES_DIR not found: {TIMESERIES_DIR}")

def ts_path_for_gauge(timeseries_dir: Path, gid: str) -> Optional[Path]:
    """Find timeseries file for gauge, returns None if not found."""
    p = timeseries_dir / f"{gid}.parquet"
    if p.exists():
        return p
    matches = list(timeseries_dir.glob(f"{gid}*.parquet"))
    if matches:
        return matches[0]
    return None

def load_discharge_series(gid: str) -> Optional[pd.Series]:
    """Load discharge timeseries, returns None if file not found."""
    p = ts_path_for_gauge(TIMESERIES_DIR, gid)
    if p is None:
        return None
    try:
        df = pd.read_parquet(p)
        if isinstance(df, pd.Series):
            s = df
        else:
            if "discharge_m3s" in df.columns:
                if "date" in df.columns:
                    date_col = "date"
                elif "time" in df.columns:
                    date_col = "time"
                else:
                    date_col = None
                if date_col is not None:
                    s = pd.Series(df["discharge_m3s"].values, index=pd.to_datetime(df[date_col]))
                else:
                    s = pd.Series(df["discharge_m3s"].values, index=pd.to_datetime(df.index))
            else:
                s = df.iloc[:, 0]
        if not isinstance(s.index, pd.DatetimeIndex):
            s.index = pd.to_datetime(s.index)
        s.index = s.index.normalize()
        if s.index.has_duplicates:
            s = s.groupby(level=0).max()
        return s.sort_index()
    except Exception as e:
        print(f"⚠️ Error reading timeseries for {gid}: {e}")
        return None

_series_cache: Dict[str, Optional[pd.Series]] = {}
_missing_gauges: set = set()

def get_series_cached(gid: str) -> Optional[pd.Series]:
    """Get cached timeseries, returns None if not available."""
    if gid not in _series_cache:
        _series_cache[gid] = load_discharge_series(gid)
        if _series_cache[gid] is None and gid not in _missing_gauges:
            _missing_gauges.add(gid)
            print(f"⚠️ No timeseries file found for gauge: {gid}")
    return _series_cache[gid]

def build_rp_grid_from_values(df_vals: pd.DataFrame, value_col: str, fill=np.nan) -> xr.DataArray:
    lats = np.sort(df_vals["lat"].unique())
    lons = np.sort(df_vals["lon"].unique())
    grid = np.full((len(lats), len(lons)), fill, dtype="float32")
    lat_index = {v:i for i,v in enumerate(lats)}
    lon_index = {v:i for i,v in enumerate(lons)}
    for _, r in df_vals.iterrows():
        grid[lat_index[r["lat"]], lon_index[r["lon"]]] = r[value_col]
    da = xr.DataArray(
        grid,
        coords={"latitude": lats, "longitude": lons},
        dims=("latitude","longitude"),
        name="return_period"
    )
    return da

def rp_from_discharge(q: float, row: pd.Series) -> float:
    return float(discharge_to_return_period_pot(
        q=q,
        u=float(row["threshold_m3s"]),
        xi=float(row["gpd_xi"]),
        sigma=float(row["gpd_sigma"]),
        lambda_u=float(row["lambda_events_per_year"]),
    ))

RP_OUT_DIR = OUT_DIR / "model" / "return_period"
RP_OUT_DIR.mkdir(parents=True, exist_ok=True)
FORCE_RP_RECOMPUTE = False  # Set True if inputs changed and cached RP needs refresh

def compute_episode_rp_maps(ep_row: pd.Series) -> Dict[str, Path]:
    ep_id = ep_row["episode_id"]
    start = pd.to_datetime(ep_row["start"])
    end = pd.to_datetime(ep_row["end"])
    window_start = start - pd.Timedelta(days=DISCHARGE_PAD_DAYS_BEFORE_EPISODE_START)
    window_end = end

    out_env = RP_OUT_DIR / f"rp__{ep_id}__envelope.nc"
    out_peak = RP_OUT_DIR / f"rp__{ep_id}__peakday.nc"

    if not FORCE_RP_RECOMPUTE and out_env.exists() and out_peak.exists():
        return {"envelope": out_env, "peakday": out_peak}
    if FORCE_RP_RECOMPUTE:
        out_env.unlink(missing_ok=True)
        out_peak.unlink(missing_ok=True)

    days = pd.date_range(window_start, window_end, freq="D")
    rp_maps = []
    daily_scores = []
    
    # Track gauges with/without data
    gauges_processed = 0
    gauges_with_data = 0

    for day in days:
        vals = []
        for _, row in evt_params.iterrows():
            gid = row["virtual_gauge_id"]
            s = get_series_cached(gid)
            
            if day == days[0]:  # Count on first day only
                gauges_processed += 1
                if s is not None:
                    gauges_with_data += 1
            
            # Handle missing timeseries
            if s is None:
                q = np.nan
            elif day in s.index:
                q = float(s.loc[day])
            else:
                q = np.nan
            
            rp = rp_from_discharge(q, row) if np.isfinite(q) else np.nan
            vals.append({"lat": row["lat"], "lon": row["lon"], "rp": rp})
        
        df_day = pd.DataFrame(vals)
        df_day.loc[df_day["rp"] < 1.0, "rp"] = np.nan  # mirror Notebook 02
        da_day = build_rp_grid_from_values(df_day, "rp", fill=np.nan).expand_dims({"time":[day]})
        rp_maps.append(da_day)

        med = float(np.nanmedian(df_day["rp"].values)) if np.isfinite(df_day["rp"].values).any() else np.nan
        daily_scores.append({"day": day, "median_rp": med})
    
    # Report gauge availability
    if gauges_processed > 0:
        pct = 100 * gauges_with_data / gauges_processed
        print(f"  📊 {ep_id}: {gauges_with_data}/{gauges_processed} gauges ({pct:.1f}%) have timeseries data")
        if gauges_with_data == 0:
            print(f"  ⚠️ {ep_id}: No gauges have timeseries - RP maps will be all-NaN")

    rp_stack = xr.concat(rp_maps, dim="time")
    rp_env = rp_stack.max(dim="time", skipna=True)

    score_df = pd.DataFrame(daily_scores).dropna()
    peak_day = score_df.sort_values("median_rp", ascending=False).iloc[0]["day"] if len(score_df)>0 else days[-1]
    rp_peak = rp_stack.sel(time=peak_day).drop_vars("time")

    rp_env.to_dataset(name="return_period").to_netcdf(out_env)
    rp_peak.to_dataset(name="return_period").to_netcdf(out_peak)

    return {"envelope": out_env, "peakday": out_peak}

print(f"\n🎯 Computing RP maps for {len(episode_df)} episodes...")
print(f"   Using {len(evt_params)} gauges within AOI")

rp_products = []
for idx, (_, ep_row) in enumerate(episode_df.iterrows(), 1):
    print(f"\n📊 Episode {idx}/{len(episode_df)}: {ep_row['episode_id']}")
    p = compute_episode_rp_maps(ep_row)
    rp_products.append({"episode_id": ep_row["episode_id"], **p})

# Summary
if _missing_gauges:
    print(f"\n⚠️ Summary: {len(_missing_gauges)} gauges had no timeseries files")

rp_df = pd.DataFrame(rp_products)
print(f"\n✅ RP computation complete: {len(rp_df)} episodes processed")
display(rp_df)

## 🔍 Optional: Diagnostic Investigation

**When to run these cells**: Only if you see unexpected gauge counts, coordinate issues,
or missing timeseries files. These cells produce detailed diagnostic output that is useful
for debugging but not required for the core validation pipeline.

**Skip to Section 7** if everything ran cleanly above.

In [ ]:
# DIAGNOSTIC CELL: Investigate the EVT parameters and gauge nomenclature
print("="*80)
print("🔍 COMPREHENSIVE DIAGNOSTIC: EVT Parameters & Gauge Investigation")
print("="*80)

# 1. Check the parquet file structure
print("\n📁 1) EVT PARAMETERS PARQUET FILE")
print(f"   Path: {EVT_PARAMS_PARQUET}")
print(f"   Exists: {EVT_PARAMS_PARQUET.exists()}")
if EVT_PARAMS_PARQUET.exists():
    import pyarrow.parquet as pq
    table = pq.read_table(EVT_PARAMS_PARQUET)
    print(f"   Rows: {table.num_rows}")
    print(f"   Size: {EVT_PARAMS_PARQUET.stat().st_size / 1024:.1f} KB")

# 2. Examine loaded data
print(f"\n📊 2) LOADED EVT_PARAMS DATA")
print(f"   Total gauges loaded: {len(evt_params)}")
print(f"   Columns: {list(evt_params.columns)}")
print(f"   Data types:\n{evt_params.dtypes}")

# 3. Sample gauge IDs to understand naming convention
print(f"\n🏷️ 3) GAUGE ID NOMENCLATURE")
sample_ids = evt_params["virtual_gauge_id"].head(10).tolist()
for i, gid in enumerate(sample_ids, 1):
    print(f"   {i:2d}. {gid}")

# Check for naming pattern
has_vg = evt_params["virtual_gauge_id"].str.startswith("VG__").any()
has_cell = evt_params["virtual_gauge_id"].str.startswith("CELL__").any()
print(f"\n   Naming pattern detected:")
print(f"     - Starts with 'VG__':   {has_vg} (Pour point extraction)")
print(f"     - Starts with 'CELL__': {has_cell} (Cell-level extraction)")

# 4. Geographic extent
print(f"\n🗺️ 4) GEOGRAPHIC EXTENT")
print(f"   Latitude range:  {evt_params['lat'].min():.4f} to {evt_params['lat'].max():.4f}")
print(f"   Longitude range: {evt_params['lon'].min():.4f} to {evt_params['lon'].max():.4f}")

# 5. AOI filtering results  
print(f"\n🎯 5) AOI FILTERING")
print(f"   Total gauges before AOI filter: {len(evt_params)}")
print(f"   Gauges after AOI filter: {len(evt_points)}")
print(f"   Percentage kept: {len(evt_points)/len(evt_params)*100:.1f}%")
print(f"   **This is why you see '22' - these are the 22 gauges WITHIN the AOI**")

# 6. Check timeseries directory
print(f"\n📂 6) TIMESERIES DIRECTORY CHECK")
print(f"   Path: {TIMESERIES_DIR}")
print(f"   Exists: {TIMESERIES_DIR.exists()}")
if TIMESERIES_DIR.exists():
    ts_files = list(TIMESERIES_DIR.glob("*.parquet"))
    print(f"   Total timeseries files: {len(ts_files)}")
    if ts_files:
        # Check naming convention of timeseries files
        sample_ts = [f.stem for f in ts_files[:5]]
        print(f"   Sample timeseries file names:")
        for ts_name in sample_ts:
            print(f"     - {ts_name}")
        
        # Check for pattern mismatch
        ts_has_vg = any(f.stem.startswith("VG__") for f in ts_files[:20])
        ts_has_cell = any(f.stem.startswith("CELL__") for f in ts_files[:20])
        print(f"\n   Timeseries naming pattern:")
        print(f"     - Starts with 'VG__':   {ts_has_vg}")
        print(f"     - Starts with 'CELL__': {ts_has_cell}")

# 7. Check for naming mismatch
print(f"\n⚠️ 7) POTENTIAL NAMING MISMATCH CHECK")
if TIMESERIES_DIR.exists():
    ts_file_stems = {f.stem for f in TIMESERIES_DIR.glob("*.parquet")}
    evt_gauge_ids = set(evt_params["virtual_gauge_id"])
    
    # Find mismatches
    in_params_not_in_ts = evt_gauge_ids - ts_file_stems
    in_ts_not_in_params = ts_file_stems - evt_gauge_ids
    
    print(f"   Gauges in EVT params but NO timeseries file: {len(in_params_not_in_ts)}")
    if in_params_not_in_ts and len(in_params_not_in_ts) <= 10:
        for gid in list(in_params_not_in_ts)[:10]:
            print(f"     - {gid}")
    
    print(f"   Timeseries files with NO corresponding EVT param: {len(in_ts_not_in_params)}")
    if len(in_ts_not_in_params) > 0:
        print(f"     **WARNING: {len(in_ts_not_in_params)} timeseries files are orphaned!**")
        if len(in_ts_not_in_params) <= 10:
            for gid in list(in_ts_not_in_params)[:10]:
                print(f"       - {gid}")

# 8. Display first few rows
print(f"\n📋 8) FIRST 5 EVT PARAMETER ROWS")
display(evt_params.head())

# 9. Display filtered gauges (within AOI)
print(f"\n📋 9) FILTERED GAUGES (WITHIN AOI) - First 5 of {len(evt_points)}")
display(evt_points[["virtual_gauge_id", "lat", "lon", "threshold_m3s", "lambda_events_per_year"]].head())

print("\n" + "="*80)
print("✅ DIAGNOSTIC COMPLETE")
print("="*80)

In [ ]:
# COORDINATE VALIDATION CELL
import matplotlib.pyplot as plt
import warnings

print("="*80)
print("🗺️ COORDINATE VALIDATION: Geographic Accuracy Check")
print("="*80)

# ============================================================================
# 1. COORDINATE PARSING VALIDATION
# ============================================================================
print("\n📍 1) COORDINATE PARSING ACCURACY")
print("-" * 80)

# Test the parsing function with sample gauges
test_samples = evt_params.head(3)
print("Testing coordinate parsing on sample gauges:")
for idx, row in test_samples.iterrows():
    gid = row["virtual_gauge_id"]
    lat_parsed = row["lat"]
    lon_parsed = row["lon"]
    
    # Re-parse to verify consistency
    lat_check, lon_check = parse_lat_lon_from_gauge_id(gid)
    
    match = (lat_parsed == lat_check) and (lon_parsed == lon_check)
    status = "✅" if match else "❌"
    
    print(f"\n{status} Gauge: {gid}")
    print(f"   Parsed:    lat={lat_parsed:.4f}, lon={lon_parsed:.4f}")
    print(f"   Re-check:  lat={lat_check:.4f}, lon={lon_check:.4f}")
    print(f"   Match: {match}")

# Check for any parsing failures
try:
    all_parsed = evt_params.apply(
        lambda r: parse_lat_lon_from_gauge_id(r["virtual_gauge_id"]),
        axis=1
    )
    print(f"\n✅ All {len(evt_params)} gauge IDs parsed successfully")
except Exception as e:
    print(f"\n❌ ERROR: Parsing failed - {e}")

# ============================================================================
# 2. GEOGRAPHIC BOUNDS VALIDATION (Philippines Region)
# ============================================================================
print("\n" + "="*80)
print("🌏 2) GEOGRAPHIC BOUNDS CHECK (Philippines Region)")
print("-" * 80)

# Philippines approximate bounds: 4°N to 21°N, 116°E to 127°E
PHL_LAT_MIN, PHL_LAT_MAX = 4.0, 21.5
PHL_LON_MIN, PHL_LON_MAX = 116.0, 127.0

lat_min, lat_max = evt_params["lat"].min(), evt_params["lat"].max()
lon_min, lon_max = evt_params["lon"].min(), evt_params["lon"].max()

print(f"\nGauge coordinate ranges:")
print(f"   Latitude:  {lat_min:.4f}° to {lat_max:.4f}°")
print(f"   Longitude: {lon_min:.4f}° to {lon_max:.4f}°")
print(f"\nPhilippines expected ranges:")
print(f"   Latitude:  {PHL_LAT_MIN}° to {PHL_LAT_MAX}°")
print(f"   Longitude: {PHL_LON_MIN}° to {PHL_LON_MAX}°")

# Check if coordinates fall within Philippines
lat_in_bounds = (lat_min >= PHL_LAT_MIN) and (lat_max <= PHL_LAT_MAX)
lon_in_bounds = (lon_min >= PHL_LON_MIN) and (lon_max <= PHL_LON_MAX)

if lat_in_bounds and lon_in_bounds:
    print(f"\n✅ All coordinates fall within Philippines bounds")
else:
    if not lat_in_bounds:
        print(f"\n⚠️ WARNING: Latitude range extends outside Philippines")
        if lat_min < PHL_LAT_MIN:
            print(f"   {(lat_min - PHL_LAT_MIN):.2f}° south of expected minimum")
        if lat_max > PHL_LAT_MAX:
            print(f"   {(lat_max - PHL_LAT_MAX):.2f}° north of expected maximum")
    
    if not lon_in_bounds:
        print(f"\n⚠️ WARNING: Longitude range extends outside Philippines")
        if lon_min < PHL_LON_MIN:
            print(f"   {(PHL_LON_MIN - lon_min):.2f}° west of expected minimum")
        if lon_max > PHL_LON_MAX:
            print(f"   {(lon_max - PHL_LON_MAX):.2f}° east of expected maximum")

# Identify any out-of-bounds gauges
out_of_bounds = evt_params[
    (evt_params["lat"] < PHL_LAT_MIN) | (evt_params["lat"] > PHL_LAT_MAX) |
    (evt_params["lon"] < PHL_LON_MIN) | (evt_params["lon"] > PHL_LON_MAX)
]
if len(out_of_bounds) > 0:
    print(f"\n⚠️ {len(out_of_bounds)} gauge(s) outside Philippines bounds:")
    for _, row in out_of_bounds.head(5).iterrows():
        print(f"   {row['virtual_gauge_id']} → lat={row['lat']:.4f}, lon={row['lon']:.4f}")
else:
    print(f"\n✅ All {len(evt_params)} gauges within Philippines geographic bounds")

# ============================================================================
# 3. CRS CONSISTENCY CHECK
# ============================================================================
print("\n" + "="*80)
print("🌐 3) CRS (Coordinate Reference System) VERIFICATION")
print("-" * 80)

print(f"\nExpected CRS: EPSG:4326 (WGS84 - World Geodetic System 1984)")
print(f"   - Units: Decimal degrees")
print(f"   - Used by: GPS, GloFAS, most global datasets")

print(f"\nevt_points GeoDataFrame CRS: {evt_points.crs}")
print(f"AOI_BOUNDARY type: {type(AOI_BOUNDARY).__name__}")

if evt_points.crs and evt_points.crs.to_string() == "EPSG:4326":
    print(f"✅ evt_points CRS is correctly set to EPSG:4326")
else:
    print(f"⚠️ WARNING: evt_points CRS mismatch!")

# Check if coordinates are in decimal degrees (typical range for lat/lon)
if (-90 <= lat_min <= 90) and (-180 <= lon_min <= 180):
    print(f"✅ Coordinate values are consistent with decimal degrees")
else:
    print(f"❌ ERROR: Coordinate values don't look like decimal degrees!")

# ============================================================================
# 4. SPATIAL FILTERING ACCURACY (AOI Intersection)
# ============================================================================
print("\n" + "="*80)
print("🎯 4) SPATIAL FILTERING ACCURACY (AOI Intersection)")
print("-" * 80)

# Verify the spatial filtering worked correctly
print(f"\nBefore AOI filtering: {len(evt_params) + len(in_params_not_in_ts) if 'in_params_not_in_ts' in globals() else 'N/A'} gauges (from full calibration)")
print(f"After AOI filtering:  {len(evt_params)} gauges")

# Check if any gauges are exactly ON the AOI boundary (edge case)
on_boundary = evt_points[evt_points.geometry.touches(AOI_BOUNDARY)]
if len(on_boundary) > 0:
    print(f"\n📍 {len(on_boundary)} gauge(s) are exactly on the AOI boundary")
    
# Verify all gauges are truly inside or intersecting AOI
inside_count = evt_points.geometry.within(AOI_BOUNDARY).sum()
intersect_count = evt_points.geometry.intersects(AOI_BOUNDARY).sum()

print(f"\nSpatial relationship to AOI:")
print(f"   Inside (within): {inside_count} gauges")
print(f"   Intersecting:    {intersect_count} gauges")
print(f"   Total selected:  {len(evt_points)} gauges")

if intersect_count == len(evt_points):
    print(f"\n✅ All selected gauges correctly intersect the AOI")
else:
    print(f"\n❌ ERROR: {len(evt_points) - intersect_count} gauges don't intersect AOI!")

# ============================================================================
# 5. COORDINATE-FILENAME MATCHING
# ============================================================================
print("\n" + "="*80)
print("📁 5) COORDINATE-FILENAME CONSISTENCY CHECK")
print("-" * 80)

# Check if timeseries filenames match the coordinates
if TIMESERIES_DIR.exists():
    sample_gauge = evt_params.iloc[0]
    gid = sample_gauge["virtual_gauge_id"]
    expected_file = TIMESERIES_DIR / f"{gid}.parquet"
    
    print(f"\nSample check:")
    print(f"   Gauge ID: {gid}")
    print(f"   Extracted lat: {sample_gauge['lat']:.4f}")
    print(f"   Extracted lon: {sample_gauge['lon']:.4f}")
    print(f"   Expected filename: {expected_file.name}")
    print(f"   File exists: {expected_file.exists()}")
    
    if expected_file.exists():
        print(f"\n✅ Filename matches coordinate convention")
    else:
        print(f"\n⚠️ WARNING: Expected file not found (may be filtered out)")

# ============================================================================
# 6. VISUAL MAP VERIFICATION
# ============================================================================
print("\n" + "="*80)
print("🗺️ 6) VISUAL MAP VERIFICATION")
print("-" * 80)

try:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))
    
    # Left plot: Full Philippines context
    ax1.set_title("Gauge Locations - Philippines Context", fontsize=14, fontweight='bold')
    
    # Plot Philippines boundary (ADM3)
    adm3_gdf.boundary.plot(ax=ax1, linewidth=0.5, color='lightgray', alpha=0.5, label='All Philippines')
    
    # L12 grid cells (from mapping)
    l12_plot = None
    if "l12_cells_sel_gdf" in globals() and l12_cells_sel_gdf is not None and len(l12_cells_sel_gdf) > 0:
        l12_plot = l12_cells_sel_gdf
    elif "l12_cells_gdf" in globals() and l12_cells_gdf is not None:
        l12_plot = l12_cells_gdf[l12_cells_gdf.intersects(AOI_BOUNDARY)].copy()
    if l12_plot is not None and not l12_plot.empty:
        l12_plot.boundary.plot(ax=ax1, linewidth=0.4, color='green', alpha=0.5, label='L12 grid')
    
    # Plot AOI boundary
    if hasattr(AOI_BOUNDARY, 'exterior'):
        # Single polygon
        xs, ys = AOI_BOUNDARY.exterior.xy
        ax1.plot(xs, ys, color='red', linewidth=2, label='AOI Boundary')
    elif hasattr(AOI_BOUNDARY, 'geoms'):
        # MultiPolygon
        for geom in AOI_BOUNDARY.geoms:
            xs, ys = geom.exterior.xy
            ax1.plot(xs, ys, color='red', linewidth=2)
        ax1.plot([], [], color='red', linewidth=2, label='AOI Boundary')
    
    # Plot virtual gauges
    evt_points.plot(ax=ax1, markersize=50, marker='x', color='blue', label=f'Gauges ({len(evt_points)})', zorder=5)
    
    # Set Philippines bounds
    ax1.set_xlim(PHL_LON_MIN, PHL_LON_MAX)
    ax1.set_ylim(PHL_LAT_MIN, PHL_LAT_MAX)
    ax1.set_xlabel('Longitude (°E)', fontsize=10)
    ax1.set_ylabel('Latitude (°N)', fontsize=10)
    ax1.legend(loc='best', fontsize=10)
    ax1.grid(True, alpha=0.3)
    
    # Right plot: Zoomed to AOI
    ax2.set_title("Gauge Locations - AOI Detail", fontsize=14, fontweight='bold')
    
    # L12 grid cells (from mapping)
    if l12_plot is not None and not l12_plot.empty:
        l12_plot.boundary.plot(ax=ax2, linewidth=0.6, color='green', alpha=0.6, label='L12 grid')
    
    # Plot AOI boundary
    if hasattr(AOI_BOUNDARY, 'exterior'):
        xs, ys = AOI_BOUNDARY.exterior.xy
        ax2.fill(xs, ys, color='lightyellow', alpha=0.3, label='AOI')
        ax2.plot(xs, ys, color='red', linewidth=2, label='AOI Boundary')
    elif hasattr(AOI_BOUNDARY, 'geoms'):
        for geom in AOI_BOUNDARY.geoms:
            xs, ys = geom.exterior.xy
            ax2.fill(xs, ys, color='lightyellow', alpha=0.3)
            ax2.plot(xs, ys, color='red', linewidth=2)
        ax2.plot([], [], color='red', linewidth=2, label='AOI Boundary')
    
    # Plot gauges
    evt_points.plot(ax=ax2, markersize=100, marker='x', color='blue', linewidth=2, label=f'Gauges ({len(evt_points)})', zorder=5)
    
    # Add gauge labels for first 5 gauges (to avoid clutter)
    for idx, row in evt_points.head(5).iterrows():
        ax2.annotate(
            f"{row['lat']:.2f},{row['lon']:.2f}",
            xy=(row.geometry.x, row.geometry.y),
            xytext=(5, 5), textcoords='offset points',
            fontsize=8, color='darkblue'
        )
    
    # Zoom to AOI with padding
    bounds = evt_points.total_bounds  # minx, miny, maxx, maxy
    padding = 0.1  # degrees
    ax2.set_xlim(bounds[0] - padding, bounds[2] + padding)
    ax2.set_ylim(bounds[1] - padding, bounds[3] + padding)
    ax2.set_xlabel('Longitude (°E)', fontsize=10)
    ax2.set_ylabel('Latitude (°N)', fontsize=10)
    ax2.legend(loc='best', fontsize=10)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n✅ Map visualization generated successfully")
    print(f"   Left: Philippines context showing all gauges relative to country")
    print(f"   Right: Zoomed to AOI showing gauge distribution detail")

except Exception as e:
    print(f"\n⚠️ Could not generate map visualization: {e}")

# ============================================================================
# SUMMARY
# ============================================================================
print("\n" + "="*80)
print("📊 COORDINATE VALIDATION SUMMARY")
print("="*80)

validation_results = {
    "Coordinate parsing": "✅ PASS" if len(evt_params) > 0 else "❌ FAIL",
    "Geographic bounds": "✅ PASS" if (lat_in_bounds and lon_in_bounds) else "⚠️ WARNING",
    "CRS verification": "✅ PASS" if evt_points.crs.to_string() == "EPSG:4326" else "⚠️ WARNING",
    "AOI filtering": "✅ PASS" if intersect_count == len(evt_points) else "❌ FAIL",
    "Coordinate format": "✅ PASS (Decimal degrees)" if (-90 <= lat_min <= 90) else "❌ FAIL",
}

for check, status in validation_results.items():
    print(f"   {check:.<50} {status}")

print(f"\n" + "="*80)
print(f"✅ COORDINATE VALIDATION COMPLETE")
print(f"   Total gauges validated: {len(evt_params)}")
print(f"   Geographic range: {lat_min:.4f}° to {lat_max:.4f}°N, {lon_min:.4f}° to {lon_max:.4f}°E")
print(f"   CRS: {evt_points.crs}")
print(f"=" * 80)

## 7) JRC flood maps (download if needed) + Depth interpolation (Notebook 02 logic)

Downloads and processes JRC global flood depth maps, then interpolates to return period grid.

**Steps**:
1. **Tile selection**: Downloads JRC tile index, selects tiles covering AOI
2. **Tile download**: Downloads flood depth maps for all 8 return periods (10-500 years)
3. **Mosaicking**: Merges tiles into single grid per RP
4. **Regridding**: Interpolates to match RP grid (bilinear method)
5. **Depth computation**: Uses CLIMADA-Petals `petals_flood_depth()` to interpolate depths

**Output**: `depth_model__validation__*.tif` files (one per episode × RP label × scenario)

**Duration**: 30-60+ minutes first time (downloading); subsequent runs use cached tiles.

**Optional**: FLOPROS flood protection scenarios (toggle `ENABLE_FLOPROS_SCENARIOS`)

In [ ]:
import requests

JRC_BASE_URL = "https://jeodpp.jrc.ec.europa.eu/ftp/jrc-opendata/CEMS-GLOFAS/flood_hazard/"
JRC_RETURN_PERIODS = [10, 20, 50, 75, 100, 200, 500]
USE_RECLASSIFIED = False

JRC_CACHE = JRC_RAW_ROOT
JRC_CACHE.mkdir(parents=True, exist_ok=True)

tile_index_path = JRC_CACHE / "tile_extents.geojson"
if not tile_index_path.exists():
    print("Downloading tile_extents.geojson ...")
    resp = requests.get(JRC_BASE_URL + "tile_extents.geojson", timeout=120)
    resp.raise_for_status()
    tile_index_path.write_bytes(resp.content)

tiles_gdf = gpd.read_file(tile_index_path).to_crs("EPSG:4326")
tiles_sel = tiles_gdf[tiles_gdf.intersects(AOI_BOUNDARY)].copy()
tiles_sel["tile_code"] = "ID" + tiles_sel["id"].astype(str) + "_" + tiles_sel["name"].astype(str)
tile_codes = sorted(tiles_sel["tile_code"].unique().tolist())
print("Selected JRC tiles:", len(tile_codes), "example:", tile_codes[:3])

def download_jrc_tile(rp: int, tile_code: str, output_dir: Path) -> Optional[Path]:
    suffix = "_depth_reclass.tif" if USE_RECLASSIFIED else "_depth.tif"
    fname = f"{tile_code}_RP{rp}{suffix}"
    out_path = output_dir / fname

    if out_path.exists() and out_path.stat().st_size > 0:
        return out_path

    url = f"{JRC_BASE_URL}RP{rp}/{fname}"
    try:
        r = requests.get(url, stream=True, timeout=300)
        r.raise_for_status()
        with open(out_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024*1024):
                if chunk:
                    f.write(chunk)
        return out_path
    except Exception as e:
        print("⚠️ Download failed:", url, "->", e)
        return None

manifest = []
for rp in JRC_RETURN_PERIODS:
    rp_dir = JRC_CACHE / f"RP{rp}"
    rp_dir.mkdir(exist_ok=True)
    ok = 0
    for tc in tile_codes:
        p = download_jrc_tile(rp, tc, rp_dir)
        if p:
            ok += 1
            manifest.append({"rp": rp, "tile_code": tc, "path": str(p)})
    print(f"RP{rp}: {ok}/{len(tile_codes)} tiles available")

if len(manifest) == 0:
    raise RuntimeError("No JRC tiles available (download failed or cache empty).")

from rasterio.merge import merge as rio_merge

flood_maps_path = OUT_DIR / "jrc" / "flood-maps_intermediate.nc"
flood_maps_path.parent.mkdir(parents=True, exist_ok=True)

if flood_maps_path.exists():
    flood_maps = xr.open_dataarray(
        flood_maps_path,
        chunks={"return_period": 1, "latitude": 1024, "longitude": 1024}
    )
    print("Using cached flood_maps (lazy/chunked):", flood_maps.shape)
else:
    suffix = "_depth_reclass.tif" if USE_RECLASSIFIED else "_depth.tif"
    flood_maps_list = []
    for rp in JRC_RETURN_PERIODS:
        rp_dir = JRC_CACHE / f"RP{rp}"
        tile_paths = [rp_dir / f"{tc}_RP{rp}{suffix}" for tc in tile_codes]
        tile_paths = [p for p in tile_paths if p.exists()]
        if not tile_paths:
            continue

        srcs = [rasterio.open(str(p)) for p in tile_paths]
        mosaic, out_trans = rio_merge(srcs)
        for s in srcs:
            s.close()

        arr = mosaic[0].astype("float32")
        h,w = arr.shape

        lons = out_trans.c + out_trans.a * (np.arange(w) + 0.5)
        lats = out_trans.f + out_trans.e * (np.arange(h) + 0.5)

        # ✓ FIX: Don't flip lats/array - rasterio expects descending lats (north→south)
        # This ensures lats.max() truly represents the northern edge for transform creation
        # (Previous flip caused vertical inversion in GeoTIFF output)

        da = xr.DataArray(
            arr,
            coords={"latitude": lats, "longitude": lons},
            dims=("latitude","longitude"),
            name="depth"
        ).expand_dims({"return_period":[rp]})

        flood_maps_list.append(da)

    flood_maps = xr.concat(flood_maps_list, dim="return_period").chunk({"return_period": 1, "latitude": 1024, "longitude": 1024})
    flood_maps.to_netcdf(flood_maps_path)
    print("Saved flood_maps:", flood_maps_path)

print("JRC flood_maps return periods:", list(flood_maps.return_period.values))

if 1 not in flood_maps.return_period.values:
    # Add RP=1 as all-NaN baseline lazily to avoid materializing the full cube in RAM
    rp1 = xr.full_like(flood_maps.isel(return_period=0, drop=True), np.nan)
    rp1 = rp1.chunk({"latitude": 1024, "longitude": 1024})
    rp1 = rp1.expand_dims({"return_period": [1]})
    flood_maps = xr.concat([rp1, flood_maps], dim="return_period").sortby("return_period")
    print("Added RP=1 null layer to flood_maps (lazy)")

DEPTH_OUT_DIR = OUT_DIR / "model" / "depth"
DEPTH_OUT_DIR.mkdir(parents=True, exist_ok=True)

REGRID_METHOD = "bilinear"

def _coord_bounds(source: xr.DataArray, coord: str) -> Tuple[float, float]:
    vals = source[coord].values
    return float(np.nanmin(vals)), float(np.nanmax(vals))

def sel_lon_lat_slice(target: xr.DataArray, source: xr.DataArray) -> xr.DataArray:
    """Select a lon/lat slice from 'target' using coordinate bounds of 'source'."""
    bounds = {}
    for coord in ["longitude", "latitude"]:
        lo, hi = _coord_bounds(source, coord)
        tgt_vals = target[coord].values
        descending = tgt_vals[0] > tgt_vals[-1]
        bounds[coord] = slice(hi, lo) if descending else slice(lo, hi)
    return target.sel(bounds)

def load_rp_da(path_nc: Path) -> xr.DataArray:
    """Load return period grid with proper spatial coordinates from NetCDF."""
    ds = xr.open_dataset(path_nc)
    da = ds["return_period"]
    # Ensure latitude and longitude coordinates are properly attached
    if "latitude" not in da.coords:
        da = da.assign_coords({"latitude": ds["latitude"]})
    if "longitude" not in da.coords:
        da = da.assign_coords({"longitude": ds["longitude"]})
    return da

def _fmt_bounds(da: xr.DataArray) -> str:
    lat_min, lat_max = _coord_bounds(da, "latitude")
    lon_min, lon_max = _coord_bounds(da, "longitude")
    return f"lat[{lat_min:.4f},{lat_max:.4f}] lon[{lon_min:.4f},{lon_max:.4f}]"

def _ensure_lon_lat_coords(da: xr.DataArray) -> xr.DataArray:
    if "lon" not in da.coords and "longitude" in da.coords:
        da = da.assign_coords(lon=da["longitude"])
    if "lat" not in da.coords and "latitude" in da.coords:
        da = da.assign_coords(lat=da["latitude"])
    return da

def _match_coord_order(source: xr.DataArray, target: xr.DataArray) -> xr.DataArray:
    source_lat_desc = source["latitude"].values[0] > source["latitude"].values[-1]
    target_lat_desc = target["latitude"].values[0] > target["latitude"].values[-1]
    if source_lat_desc != target_lat_desc:
        source = source.sortby("latitude", ascending=not target_lat_desc)
    source_lon_desc = source["longitude"].values[0] > source["longitude"].values[-1]
    target_lon_desc = target["longitude"].values[0] > target["longitude"].values[-1]
    if source_lon_desc != target_lon_desc:
        source = source.sortby("longitude", ascending=not target_lon_desc)
    return source

def _da_stats(da: xr.DataArray, label: str) -> None:
    arr = da.values
    finite = np.isfinite(arr)
    n = int(finite.sum())
    total = int(arr.size)
    if n > 0:
        mn = float(np.nanmin(arr))
        mx = float(np.nanmax(arr))
        print(f"{label}: finite={n}/{total} min={mn:.3f} max={mx:.3f}")
    else:
        print(f"{label}: finite=0/{total}")

def _sanitize_rp(da: xr.DataArray) -> xr.DataArray:
    # RP values below 1 are invalid for flood depth logic; treat as missing
    return da.where(da >= 1.0)

def apply_flopros_if_needed(rp_regrid: xr.DataArray, apply: bool, flopros_root: Path) -> xr.DataArray:
    if not apply:
        return rp_regrid
    try:
        flopros_shp = flopros_root / "FLOPROS_shp_V1" / "FLOPROS_shp_V1.shp"
        if not flopros_shp.exists():
            print("Downloading FLOPROS database ...")
            download_flopros_database(str(flopros_root))
        flopros_gdf = gpd.read_file(flopros_shp).to_crs("EPSG:4326")
        overlap = int(flopros_gdf.intersects(AOI_BOUNDARY).sum())
        print(f"FLOPROS AOI overlap polygons: {overlap}")

        protected_list = []
        for i, ep in enumerate(rp_regrid.event.values):
            rp_event = _sanitize_rp(rp_regrid.isel(event=i).copy(deep=True))
            _da_stats(rp_event, f"FLOPROS pre-mask {ep}")
            if overlap == 0:
                print("FLOPROS: no AOI overlap; skipping protection mask")
                rp_prot = rp_event
            else:
                rp_prot = petals_apply_flopros(flopros_gdf, rp_event, layer="MerL_Riv")
            _da_stats(rp_prot, f"FLOPROS post-mask {ep}")
            protected_list.append(rp_prot.expand_dims({"event":[ep]}))
        return xr.concat(protected_list, dim="event")
    except Exception as e:
        print("⚠️ FLOPROS failed; continuing without protection:", e)
        return rp_regrid

_CACHED_REGRIDDER = None
_CACHED_REGRIDDER_SHAPE = None

def compute_depth_from_rp(rp_da: xr.DataArray, ep_id: str, rp_label: str, apply_flopros: bool) -> Optional[Path]:
    # Create properly-structured 3D array with event dimension
    # Ensure dimensions are named: (event, latitude, longitude)
    rp_da_3d = rp_da.expand_dims({"event": [ep_id]})
    rp_da_3d = _ensure_lon_lat_coords(rp_da_3d)
    rp_da_3d = _sanitize_rp(rp_da_3d)

    # Verify coordinates before regridding
    assert "latitude" in rp_da_3d.coords or "latitude" in rp_da_3d.dims, f"Missing latitude coordinate/dimension"
    assert "longitude" in rp_da_3d.coords or "longitude" in rp_da_3d.dims, f"Missing longitude coordinate/dimension"

    # Crop flood_maps to AOI extent using return period bounds (matches notebook 2 approach)
    flood_maps_sel = sel_lon_lat_slice(flood_maps, rp_da_3d)
    flood_maps_sel = _ensure_lon_lat_coords(flood_maps_sel)
    if "latitude" not in flood_maps_sel.dims or "longitude" not in flood_maps_sel.dims:
        raise ValueError(f"flood_maps_sel missing lat/lon dims: {flood_maps_sel.dims}")
    if flood_maps_sel.sizes.get("latitude", 0) == 0 or flood_maps_sel.sizes.get("longitude", 0) == 0:
        print("⚠️ Empty flood_maps selection for regrid")
        print("   rp_da_3d:", _fmt_bounds(rp_da_3d))
        print("   flood_maps:", _fmt_bounds(flood_maps))
        raise ValueError("No overlap between return-period grid and JRC flood maps. Check AOI and data bounds.")

    rp_da_3d = _match_coord_order(rp_da_3d, flood_maps_sel)

    # Guard: xESMF bilinear requires >= 2 points in each dimension.
    # A degenerate source RP grid (single row or column) raises ESMC_FieldRegridStore rc=545.
    # Return None so the calling loop can skip this episode cleanly.
    _nlat_src = rp_da_3d.sizes.get("latitude", 0)
    _nlon_src = rp_da_3d.sizes.get("longitude", 0)
    if _nlat_src < 2 or _nlon_src < 2:
        print(f"⚠️  Skipping regrid for {ep_id}/{rp_label}: source grid too small "
              f"({_nlat_src}×{_nlon_src}). Need ≥2 in each dim for bilinear.")
        return None

    # Regrid to match flood_maps grid (cache regridder across episodes with same source shape)
    global _CACHED_REGRIDDER, _CACHED_REGRIDDER_SHAPE
    _src_shape = (_nlat_src, _nlon_src)
    if _CACHED_REGRIDDER is not None and _CACHED_REGRIDDER_SHAPE != _src_shape:
        _CACHED_REGRIDDER = None
    rp_regrid, _CACHED_REGRIDDER = petals_regrid(
        rp_da_3d, flood_maps_sel, method=REGRID_METHOD,
        regridder=_CACHED_REGRIDDER, return_regridder=True,
    )
    _CACHED_REGRIDDER_SHAPE = _src_shape
    rp_regrid = _sanitize_rp(rp_regrid)
    rp_regrid = apply_flopros_if_needed(rp_regrid, apply_flopros, flopros_root=OUT_DIR / "jrc")
    depth = petals_flood_depth(rp_regrid, flood_maps_sel).isel(event=0)

    scenario = "FLOPROS" if apply_flopros else "NoProt"
    out_tif = DEPTH_OUT_DIR / f"depth_model__validation__{ep_id}__{rp_label}__{scenario}.tif"

    lats = depth.latitude.values
    lons = depth.longitude.values
    res_lon = float(np.mean(np.diff(lons)))
    res_lat = float(np.mean(np.diff(lats)))
    west = float(lons.min() - res_lon/2)
    north = float(lats.max() + abs(res_lat)/2)

    # ✓ Transform: lats should be descending (north→south) so lats.max() is truly north
    # from_origin expects (west, north) corner with positive pixel sizes
    transform = rasterio.transform.from_origin(west, north, res_lon, abs(res_lat))

    meta = {
        "driver":"GTiff",
        "height": depth.shape[0],
        "width": depth.shape[1],
        "count":1,
        "dtype":"float32",
        "crs":"EPSG:4326",
        "transform": transform,
        "nodata": np.nan
    }
    with rasterio.open(out_tif, "w", **meta) as dst:
        dst.write(depth.values.astype("float32"), 1)
    return out_tif

depth_products = []
for _, row in rp_df.iterrows():
    ep_id = row["episode_id"]
    for rp_label, path_nc in [("envelope", Path(row["envelope"])), ("peakday", Path(row["peakday"]))]:
        rp_da = load_rp_da(path_nc)
        scenarios = [False] + ([True] if ENABLE_FLOPROS_SCENARIOS else [])
        for do_flopros in scenarios:
            try:
                out_tif = compute_depth_from_rp(rp_da, ep_id, rp_label, apply_flopros=do_flopros)
            except ValueError as _esmf_err:
                if "ESMC_FieldRegridStore" in str(_esmf_err) or "rc = 545" in str(_esmf_err):
                    print(f"⚠️  ESMF regrid failed for {ep_id}/{rp_label} "
                          f"(rc=545, degenerate grid) — episode skipped.")
                    out_tif = None
                else:
                    raise
            if out_tif is None:
                continue
            depth_products.append({
                "episode_id": ep_id,
                "rp_label": rp_label,
                "scenario": "FLOPROS" if do_flopros else "NoProt",
                "depth_tif": str(out_tif)
            })

depth_df = pd.DataFrame(depth_products)
display(depth_df)

## 8) Hazard extent validation (observed vs modelled)

**Core validation step**: Compares modeled vs. observed flood extent quantitatively.

**Steps**:
1. **Reprojection**: Transforms observed GFM extent to model grid (nearest neighbor)
2. **Confusion matrix** at multiple depth thresholds (0.05-1.0 m):
   - **TP** (True Positive): Correctly predicted flood
   - **FP** (False Positive): Over-predicted (false alarm)
   - **FN** (False Negative): Under-predicted (missed)
   - **TN** (True Negative): Correctly predicted non-flood
3. **Derived metrics**: Precision, Recall, F1, IoU, Bias
4. **Difference maps**: 4-class maps (TP=1, FP=2, FN=3, TN=0)

**Output**: `hazard_extent_metrics.csv` + diff maps in `maps_for_dashboard/`

**Metrics guide**: See [GLOSSARY - Validation Metrics](../../docs/technical/GLOSSARY.md#validation--metrics-notebook-03)

In [ ]:

METRICS_OUT_DIR = OUT_DIR / "metrics"
MAPS_OUT_DIR = OUT_DIR / "maps_for_dashboard"
METRICS_OUT_DIR.mkdir(parents=True, exist_ok=True)
MAPS_OUT_DIR.mkdir(parents=True, exist_ok=True)

def read_raster(path: Path) -> Tuple[np.ndarray, dict]:
    with rasterio.open(path) as src:
        arr = src.read(1)
        meta = src.meta.copy()
    return arr, meta

def reproject_to_match(src_arr, src_meta, dst_meta, dst_shape) -> np.ndarray:
    dst = np.full(dst_shape, np.nan, dtype="float32")
    reproject(
        source=src_arr.astype("float32"),
        destination=dst,
        src_transform=src_meta["transform"],
        src_crs=src_meta.get("crs", "EPSG:4326"),
        dst_transform=dst_meta["transform"],
        dst_crs=dst_meta.get("crs", "EPSG:4326"),
        resampling=Resampling.nearest,
        src_nodata=src_meta.get("nodata", 255),
        dst_nodata=np.nan
    )
    return dst

def confusion_metrics(obs: np.ndarray, mod: np.ndarray, valid_mask: np.ndarray) -> Dict[str,float]:
    o = obs[valid_mask].astype(int)
    m = mod[valid_mask].astype(int)
    tp = int(((o==1) & (m==1)).sum())
    fp = int(((o==0) & (m==1)).sum())
    fn = int(((o==1) & (m==0)).sum())
    tn = int(((o==0) & (m==0)).sum())
    precision = tp / (tp+fp) if (tp+fp)>0 else np.nan
    recall = tp / (tp+fn) if (tp+fn)>0 else np.nan
    f1 = 2*precision*recall/(precision+recall) if np.isfinite(precision) and np.isfinite(recall) and (precision+recall)>0 else np.nan
    iou = tp / (tp+fp+fn) if (tp+fp+fn)>0 else np.nan
    bias = (tp+fp) / (tp+fn) if (tp+fn)>0 else np.nan
    return {"tp":tp,"fp":fp,"fn":fn,"tn":tn,"precision":precision,"recall":recall,"f1":f1,"iou":iou,"bias":bias}

def make_diff_map(obs_bin, mod_bin, valid_mask) -> np.ndarray:
    out = np.full(obs_bin.shape, 255, dtype=np.uint8)
    out[valid_mask] = 0
    out[(valid_mask) & (obs_bin==1) & (mod_bin==1)] = 1  # TP
    out[(valid_mask) & (obs_bin==0) & (mod_bin==1)] = 2  # FP
    out[(valid_mask) & (obs_bin==1) & (mod_bin==0)] = 3  # FN
    return out

# Template grid from first depth raster
depth0_path = Path(depth_df["depth_tif"].iloc[0])
depth0_arr, depth0_meta = read_raster(depth0_path)
dst_shape = depth0_arr.shape

# Validate projection metadata
print("\n🔍 **CRS Validation & Projection Metadata**")
depth0_crs = depth0_meta.get("crs", "EPSG:4326")
print(f"Depth map CRS: {depth0_crs}")
print(f"Depth map shape: {dst_shape}")
print(f"Depth map transform (affine): {depth0_meta['transform']}")

# Verify depth map bounds (sanity check)
trans = depth0_meta['transform']
west = trans.c
north = trans.f
east = west + trans.a * depth0_meta['width']
south = north + trans.e * depth0_meta['height']
print(f"Depth map bounds (west, south, east, north): ({west:.4f}, {south:.4f}, {east:.4f}, {north:.4f})")

# Cache observed (episode max) reprojected to JRC grid
obs_reproj_cache = {}
print("\n🔍 **Observed Flood Map Reprojection**")
for _, ep in episode_df.iterrows():
    ep_id = ep["episode_id"]
    obs_path = Path(ep["path"])
    obs_arr, obs_meta = read_raster(obs_path)
    
    # Validate obs CRS
    obs_crs = obs_meta.get("crs", "EPSG:4326")
    if obs_crs != depth0_crs:
        print(f"⚠️ Episode {ep_id}: CRS mismatch - obs={obs_crs}, depth={depth0_crs}")
    
    obs_float = np.where(obs_arr == 255, np.nan, obs_arr.astype("float32"))
    obs_reproj = reproject_to_match(obs_float, obs_meta, depth0_meta, dst_shape)
    obs_bin = np.where(np.isfinite(obs_reproj) & (obs_reproj>=0.5), 1, 0).astype(np.uint8)
    valid_mask = np.isfinite(obs_reproj)
    obs_reproj_cache[ep_id] = (obs_bin, valid_mask)
    print(f"✓ {ep_id}: Reprojected to depth grid (shape={obs_reproj.shape}, valid pixels={valid_mask.sum():,})")

metrics_rows = []
for _, row in depth_df.iterrows():
    ep_id = row["episode_id"]
    rp_label = row["rp_label"]
    scenario = row["scenario"]

    depth_path = Path(row["depth_tif"])
    depth_arr, depth_meta = read_raster(depth_path)
    depth_arr = depth_arr.astype("float32")
    if depth_arr.shape != dst_shape:
        depth_arr = reproject_to_match(depth_arr, depth_meta, depth0_meta, dst_shape)

    obs_bin, valid_mask = obs_reproj_cache[ep_id]

    for thr in DEPTH_THRESHOLDS_FOR_METRICS:
        mod_bin = np.where(np.isfinite(depth_arr) & (depth_arr >= thr), 1, 0).astype(np.uint8)
        met = confusion_metrics(obs_bin, mod_bin, valid_mask)
        metrics_rows.append({
            "episode_id": ep_id,
            "rp_label": rp_label,
            "scenario": scenario,
            "depth_threshold_m": thr,
            **met
        })

        diff = make_diff_map(obs_bin, mod_bin, valid_mask)
        diff_path = MAPS_OUT_DIR / f"diff__{ep_id}__{rp_label}__{scenario}__thr{thr:.2f}.tif"
        meta = depth_meta.copy()
        meta.update({"dtype": rasterio.uint8, "count":1, "nodata":255})
        with rasterio.open(diff_path, "w", **meta) as dst:
            dst.write(diff, 1)

metrics_df = pd.DataFrame(metrics_rows)
metrics_csv = METRICS_OUT_DIR / "hazard_extent_metrics.csv"
metrics_df.to_csv(metrics_csv, index=False)
print("Saved:", metrics_csv)
display(metrics_df.head(20))


In [ ]:
# Analysis: Best scenario & threshold per episode
import pandas as pd

# Load metrics
metrics_csv = METRICS_OUT_DIR / "hazard_extent_metrics.csv"
metrics_df = pd.read_csv(metrics_csv)

# Filter to envelope RP label (most relevant for episode-wide comparison)
m_env = metrics_df[metrics_df["rp_label"] == "envelope"].copy()

print("="*80)
print("📊 BEST SCENARIO & THRESHOLD PER EPISODE")
print("="*80)

for ep_id in sorted(m_env["episode_id"].unique()):
    print(f"\n{'='*80}")
    print(f"EPISODE: {ep_id}")
    print('='*80)
    
    ep_data = m_env[m_env["episode_id"] == ep_id].copy()
    
    # Rank by F1 score (best overall metric)
    ep_data_sorted = ep_data.sort_values("f1", ascending=False)
    
    print("\n🥇 TOP 5 CONFIGURATIONS (by F1 score):")
    print("-" * 80)
    
    top5 = ep_data_sorted.head(5)
    for idx, (_, row) in enumerate(top5.iterrows(), 1):
        print(f"\n{idx}. Rank {idx}")
        print(f"   Scenario:  {row['scenario']}")
        print(f"   Threshold: {row['depth_threshold_m']:.2f} m")
        print(f"   F1:        {row['f1']:.3f}")
        print(f"   IoU:       {row['iou']:.3f}")
        print(f"   Precision: {row['precision']:.3f}")
        print(f"   Recall:    {row['recall']:.3f}")
        print(f"   Bias:      {row['bias']:.3f}")
        print(f"   TP/FP/FN:  {int(row['tp'])}/{int(row['fp'])}/{int(row['fn'])}")
    
    # Compare scenarios at common threshold (0.20m)
    print(f"\n📍 SCENARIO COMPARISON at 0.20m threshold:")
    print("-" * 80)
    comp = ep_data[ep_data["depth_threshold_m"] == 0.20].copy()
    for _, row in comp.iterrows():
        print(f"   {row['scenario']:8s}: F1={row['f1']:.3f}, IoU={row['iou']:.3f}, Bias={row['bias']:.3f}")

# Overall summary
print("\n\n" + "="*80)
print("🎯 OVERALL SUMMARY")
print("="*80)

# Best by episode
best_per_ep = []
for ep_id in sorted(m_env["episode_id"].unique()):
    best_row = m_env[m_env["episode_id"] == ep_id].sort_values("f1", ascending=False).iloc[0]
    best_per_ep.append({
        "episode_id": ep_id,
        "best_scenario": best_row["scenario"],
        "best_threshold": best_row["depth_threshold_m"],
        "f1": best_row["f1"],
        "iou": best_row["iou"]
    })

summary_df = pd.DataFrame(best_per_ep)
display(summary_df)

# Scenario performance aggregation
print("\n📊 SCENARIO PERFORMANCE ACROSS ALL EPISODES:")
print("-" * 80)
scenario_stats = m_env.groupby("scenario").agg({
    "f1": ["mean", "median", "max"],
    "iou": ["mean", "median", "max"],
    "bias": ["mean", "median"]
}).round(3)
display(scenario_stats)

print("\n💡 INTERPRETATION:")
print("   - F1 > 0.70: Excellent agreement")
print("   - F1 0.50-0.70: Good agreement")
print("   - F1 < 0.50: Poor agreement (investigate)")
print("   - Bias > 1.0: Over-prediction (too much flood)")
print("   - Bias < 1.0: Under-prediction (missed flood)")

In [ ]:
# ------------------------------------------------------------
# EXTRA VALIDATION METRICS + "PLATEAU/KNEE" THRESHOLD PICKING
# This cell runs after Cell 25 (hazard extent validation metrics)
# 
# REQUIRED VARIABLES (from earlier cells):
#   - metrics_df: DataFrame with columns [episode_id, rp_label, scenario, 
#     depth_threshold_m, tp, fp, fn, tn, precision, recall, f1, iou, bias]
#     Created in Cell 24 (lines 1770-1894)
#   
#   - depth_df: DataFrame with columns [episode_id, rp_label, scenario, depth_tif]
#     Created in Cell 22 (lines 1471-1748)
#   
#   - obs_reproj_cache: Dict mapping episode_id -> (obs_bin, valid_mask)
#     where obs_bin is binary observed flood (0/1) and valid_mask is boolean array
#     Created in Cell 24 (lines 1770-1894)
#   
#   - DEPTH_THRESHOLDS_FOR_METRICS: List[float] of depth thresholds in meters
#     Defined in Cell 11 (lines 298-329)
#
# REQUIRED IMPORTS (from Cell 3):
#   - numpy as np, pandas as pd, matplotlib.pyplot as plt
#   - Path, rasterio, IPython.display
#   - scipy (optional, for tolerant metrics with buffer)
#
# OUTPUTS:
#   - agg: DataFrame with aggregated macro/micro metrics per (rp_label, scenario, depth_threshold_m)
#   - best_df: DataFrame with recommended thresholds per scenario
#   - tol_df, tol_micro, tol_best_df: Optional tolerant metrics (if RUN_TOLERANT=True)
#   - Multiple matplotlib plots showing F1 curves
# ------------------------------------------------------------

# Imports already available from earlier cells (Cell 3):
# - numpy as np, pandas as pd, IPython.display.display, matplotlib.pyplot as plt
# - Path, rasterio, scipy (if installed)

# Validate required variables are available
print("🔍 Validating required variables from earlier cells...")

_required_vars = {
    "metrics_df": "DataFrame from Cell 24 (hazard extent metrics)",
    "depth_df": "DataFrame from Cell 22 (depth products)",
    "obs_reproj_cache": "Dict from Cell 24 (reprojected observed extents)",
    "DEPTH_THRESHOLDS_FOR_METRICS": "List from Cell 11 (depth thresholds)"
}

_missing_vars = []
for var_name, var_desc in _required_vars.items():
    if var_name not in globals():
        _missing_vars.append(f"  ❌ {var_name}: {var_desc}")
    else:
        print(f"  ✅ {var_name}: Found")

if _missing_vars:
    print("\n⚠️ ERROR: Missing required variables. Please run earlier cells first:")
    for msg in _missing_vars:
        print(msg)
    raise RuntimeError("Missing required variables. Run cells 1-25 before running this cell.")
else:
    print("\n✅ All required variables found. Proceeding with analysis...")


# ---- Helper: read_raster (already defined in Cell 24, redefined here for clarity)
def read_raster(path: Path) -> Tuple[np.ndarray, dict]:
    """Read raster file and return array + metadata."""
    with rasterio.open(path) as src:
        arr = src.read(1)
        meta = src.meta.copy()
    return arr, meta

# ---- Helpers
def _safe_div(a, b):
    return a / b if b else np.nan

def metrics_from_counts(tp, fp, fn, tn):
    precision = _safe_div(tp, tp + fp)
    recall    = _safe_div(tp, tp + fn)
    f1        = _safe_div(2 * precision * recall, (precision + recall)) if np.isfinite(precision) and np.isfinite(recall) else np.nan
    iou       = _safe_div(tp, tp + fp + fn)
    acc       = _safe_div(tp + tn, tp + fp + fn + tn)
    spec      = _safe_div(tn, tn + fp)              # specificity (TNR)
    bal_acc   = np.nanmean([recall, spec])          # balanced accuracy
    mcc_den   = float((tp+fp)*(tp+fn)*(tn+fp)*(tn+fn))
    mcc       = ((tp*tn - fp*fn) / np.sqrt(mcc_den)) if mcc_den > 0 else np.nan
    # Cohen's kappa
    tot = tp + fp + fn + tn
    if tot > 0:
        po = (tp + tn) / tot
        pe = (((tp+fp)*(tp+fn) + (fn+tn)*(fp+tn)) / (tot*tot))
        kappa = (po - pe) / (1 - pe) if (1 - pe) != 0 else np.nan
    else:
        kappa = np.nan
    bias = _safe_div(tp + fp, tp + fn)
    return dict(precision=precision, recall=recall, f1=f1, iou=iou, accuracy=acc,
                specificity=spec, balanced_accuracy=bal_acc, mcc=mcc, kappa=kappa, bias=bias)

def select_plateau_threshold(df, metric_col="f1_micro", delta=0.02):
    """
    Pick the LOWEST depth threshold that is within delta of the maximum metric.
    This avoids choosing an overly high threshold when performance is on a plateau.
    """
    d = df.sort_values("depth_threshold_m").copy()
    mmax = d[metric_col].max()
    keep = d[d[metric_col] >= (mmax - delta)]
    if keep.empty:
        return np.nan
    return keep["depth_threshold_m"].min()

# ---- 1) Aggregate metrics: macro (mean of per-episode F1) and micro (sum counts → F1)
group_cols = ["rp_label", "scenario", "depth_threshold_m"]

# Macro averages (per-episode metrics averaged)
agg_macro = (metrics_df
             .groupby(group_cols, as_index=False)
             .agg(
                 f1_macro=("f1", "mean"),
                 iou_macro=("iou", "mean"),
                 precision_macro=("precision", "mean"),
                 recall_macro=("recall", "mean"),
                 bias_macro=("bias", "mean"),
                 n_episodes=("episode_id", "nunique")
             ))

# Micro averages (sum TP/FP/FN/TN then recompute metrics)
agg_micro_counts = (metrics_df
                    .groupby(group_cols, as_index=False)
                    .agg(tp=("tp","sum"), fp=("fp","sum"), fn=("fn","sum"), tn=("tn","sum")))

extra = agg_micro_counts.apply(lambda r: pd.Series(metrics_from_counts(int(r.tp), int(r.fp), int(r.fn), int(r.tn))), axis=1)
agg_micro = pd.concat([agg_micro_counts, extra.add_suffix("_micro")], axis=1)

# Merge macro+micro for convenience
agg = pd.merge(agg_micro, agg_macro, on=group_cols, how="left")

print("✅ Aggregated metrics table (macro + micro):")
display(agg.head(20))

# ---- 2) Plateau/knee threshold selection (per scenario + rp_label)
DELTA_F1 = 0.02  # tweakable: 0.01–0.03 typical
best_rows = []
for (rp_label, scenario), sub in agg.groupby(["rp_label", "scenario"]):
    sub = sub.sort_values("depth_threshold_m")
    thr_best = sub.loc[sub["f1_micro"].idxmax(), "depth_threshold_m"]
    thr_plateau = select_plateau_threshold(sub, metric_col="f1_micro", delta=DELTA_F1)

    best_rows.append({
        "rp_label": rp_label,
        "scenario": scenario,
        "thr_best_f1_micro": float(thr_best),
        f"thr_plateau_f1_micro_delta{DELTA_F1:.2f}": float(thr_plateau),
        "max_f1_micro": float(sub["f1_micro"].max()),
    })

best_df = pd.DataFrame(best_rows).sort_values(["scenario", "rp_label"])
print("\n🎯 Recommended thresholds (strict micro-F1):")
display(best_df)

# Optional: plot F1_micro curves (one plot per scenario)
import matplotlib.pyplot as plt

for scenario, sub_s in agg.groupby("scenario"):
    plt.figure()
    for rp_label, sub_rp in sub_s.groupby("rp_label"):
        sub_rp = sub_rp.sort_values("depth_threshold_m")
        plt.plot(sub_rp["depth_threshold_m"], sub_rp["f1_micro"], marker="o", label=str(rp_label))
    plt.xlabel("Depth threshold (m)")
    plt.ylabel("F1 (micro, strict)")
    plt.title(f"F1_micro vs depth threshold — scenario={scenario}")
    plt.legend()
    plt.grid(True)
    plt.show()

# ---- 3) Tolerant / buffered F1 (handles small misregistration & SAR edge noise)
# This RE-RUNS the raster comparisons (needed because tolerance changes TP/FP/FN).
# If you don't want the extra compute, set RUN_TOLERANT = False.
RUN_TOLERANT = True
TOL_RADII_PIX = [1, 2]  # 1–2 pixels is usually enough

if RUN_TOLERANT:
    try:
        from scipy.ndimage import binary_dilation
    except Exception as e:
        raise ImportError("scipy is required for buffered/tolerant metrics. Install scipy or set RUN_TOLERANT=False") from e

    def disk_structure(radius: int):
        y, x = np.ogrid[-radius:radius+1, -radius:radius+1]
        return (x*x + y*y) <= radius*radius

    def confusion_metrics_tolerant(obs_bin, mod_bin, valid_mask, radius_pix=1):
        """
        Symmetric tolerance:
        - A model positive counts as TP if it's within radius of any observed positive (obs dilated).
        - An observed positive counts as FN only if there's no model positive within radius (model dilated).
        """
        obs_pos = (obs_bin == 1) & valid_mask
        mod_pos = (mod_bin == 1) & valid_mask

        if radius_pix <= 0:
            # fall back to strict
            o = obs_bin[valid_mask].astype(int)
            m = mod_bin[valid_mask].astype(int)
            tp = int(((o==1) & (m==1)).sum())
            fp = int(((o==0) & (m==1)).sum())
            fn = int(((o==1) & (m==0)).sum())
            tn = int(((o==0) & (m==0)).sum())
            out = metrics_from_counts(tp, fp, fn, tn)
            out.update(dict(tp=tp, fp=fp, fn=fn, tn=tn))
            return out

        st = disk_structure(radius_pix)
        obs_buf = binary_dilation(obs_pos, structure=st)
        mod_buf = binary_dilation(mod_pos, structure=st)

        tp = int((mod_pos & obs_buf).sum())
        fp = int((mod_pos & (~obs_buf) & valid_mask).sum())
        fn = int((obs_pos & (~mod_buf)).sum())
        tn = int(((~obs_pos) & (~mod_pos) & valid_mask).sum())

        out = metrics_from_counts(tp, fp, fn, tn)
        out.update(dict(tp=tp, fp=fp, fn=fn, tn=tn))
        return out

    tol_rows = []
    for _, row in depth_df.iterrows():
        ep_id = row["episode_id"]
        rp_label = row["rp_label"]
        scenario = row["scenario"]

        depth_path = Path(row["depth_tif"])
        depth_arr, depth_meta_tol = read_raster(depth_path)
        depth_arr = depth_arr.astype("float32")
        if depth_arr.shape != dst_shape:
            depth_arr = reproject_to_match(depth_arr, depth_meta_tol, depth0_meta, dst_shape)

        obs_bin, valid_mask = obs_reproj_cache[ep_id]

        for thr in DEPTH_THRESHOLDS_FOR_METRICS:
            mod_bin = np.where(np.isfinite(depth_arr) & (depth_arr >= thr), 1, 0).astype(np.uint8)

            for rad in TOL_RADII_PIX:
                met = confusion_metrics_tolerant(obs_bin, mod_bin, valid_mask, radius_pix=rad)
                tol_rows.append({
                    "episode_id": ep_id,
                    "rp_label": rp_label,
                    "scenario": scenario,
                    "depth_threshold_m": thr,
                    "tol_radius_pix": rad,
                    **{k: met[k] for k in ["tp","fp","fn","tn"]},
                    "precision_tol": met["precision"],
                    "recall_tol": met["recall"],
                    "f1_tol": met["f1"],
                    "iou_tol": met["iou"],
                    "bias_tol": met["bias"],
                    "mcc_tol": met["mcc"],
                    "kappa_tol": met["kappa"],
                })

    tol_df = pd.DataFrame(tol_rows)

    # Aggregate tolerant results (micro)
    tol_micro = (tol_df.groupby(["rp_label","scenario","depth_threshold_m","tol_radius_pix"], as_index=False)
                 .agg(tp=("tp","sum"), fp=("fp","sum"), fn=("fn","sum"), tn=("tn","sum")))

    extra_tol = tol_micro.apply(lambda r: pd.Series(metrics_from_counts(int(r.tp), int(r.fp), int(r.fn), int(r.tn))), axis=1)
    tol_micro = pd.concat([tol_micro, extra_tol.add_suffix("_micro_tol")], axis=1)

    print("\n✅ Tolerant (buffered) micro metrics:")
    display(tol_micro.head(20))

    # Plateau selection for tolerant F1 (per radius)
    tol_best_rows = []
    for (rp_label, scenario, rad), sub in tol_micro.groupby(["rp_label","scenario","tol_radius_pix"]):
        sub = sub.sort_values("depth_threshold_m")
        thr_best = sub.loc[sub["f1_micro_tol"].idxmax(), "depth_threshold_m"]
        thr_plateau = select_plateau_threshold(sub.rename(columns={"f1_micro_tol":"f1_micro"}),
                                               metric_col="f1_micro", delta=DELTA_F1)
        tol_best_rows.append({
            "rp_label": rp_label,
            "scenario": scenario,
            "tol_radius_pix": int(rad),
            "thr_best_f1_micro_tol": float(thr_best),
            f"thr_plateau_f1_micro_tol_delta{DELTA_F1:.2f}": float(thr_plateau),
            "max_f1_micro_tol": float(sub["f1_micro_tol"].max()),
        })
    tol_best_df = pd.DataFrame(tol_best_rows).sort_values(["scenario","rp_label","tol_radius_pix"])
    print("\n🎯 Recommended thresholds (tolerant micro-F1):")
    display(tol_best_df)

    # Optional plot: tolerant curves
    for scenario, sub_s in tol_micro.groupby("scenario"):
        for rad, sub_rad in sub_s.groupby("tol_radius_pix"):
            plt.figure()
            for rp_label, sub_rp in sub_rad.groupby("rp_label"):
                sub_rp = sub_rp.sort_values("depth_threshold_m")
                plt.plot(sub_rp["depth_threshold_m"], sub_rp["f1_micro_tol"], marker="o", label=str(rp_label))
            plt.xlabel("Depth threshold (m)")
            plt.ylabel(f"F1 (micro, tolerant r={rad}px)")
            plt.title(f"Tolerant F1_micro vs depth threshold — scenario={scenario}, r={rad}px")
            plt.legend()
            plt.grid(True)
            plt.show()

# ---- Final Summary ----
print("\n" + "="*80)
print("📊 EXTRA VALIDATION METRICS - SUMMARY")
print("="*80)
print(f"\n✅ Computed aggregated metrics (macro + micro):")
print(f"   - {len(agg)} rows (combinations of rp_label × scenario × depth_threshold)")
print(f"   - Metrics: F1, IoU, Precision, Recall, Bias (macro and micro)")
print(f"\n✅ Identified optimal thresholds:")
print(f"   - Strict (max F1): best_df")
print(f"   - Plateau/knee (F1 within Δ={DELTA_F1}): best_df")
if RUN_TOLERANT:
    print(f"\n✅ Computed tolerant (buffered) metrics:")
    print(f"   - {len(tol_df)} rows (with tolerance radii: {TOL_RADII_PIX})")
    print(f"   - Accounts for small spatial misalignment between model and observations")
    print(f"   - Tolerant thresholds: tol_best_df")
else:
    print(f"\n⏭️  Tolerant metrics: SKIPPED (RUN_TOLERANT=False)")
print(f"\n📈 Plots generated: F1 vs depth threshold curves")
print(f"   - One plot per scenario (strict metrics)")
if RUN_TOLERANT:
    print(f"   - Additional plots for tolerant metrics (per radius)")
print("\n" + "="*80)
print("✅ ANALYSIS COMPLETE")
print("="*80)


## 9) Affected population proxy (WorldPop × depth ≥ threshold)

Estimates affected population as impact proxy (not true impact modeling).

**Steps**:
1. **Reprojection**: Regrid WorldPop to model grid (bilinear interpolation)
2. **Mass balance check**: Compare native vs. reprojected total population (warn if >10% change)
3. **Exposure calculation**: For each depth threshold:
   - Identify flooded areas (depth ≥ threshold)
   - Sum WorldPop within flood extent
   - Mask to ADM3 municipality boundaries
4. **GFM comparison**: Same calculation for observed extent (modeled vs. observed pop)

**Output**: `affected_population_by_adm3.csv`

**Columns**: episode_id, scenario, depth_threshold, adm3_name, affected_pop, observed_pop

**Note**: Population is a proxy only. Use to identify areas of concern, not absolute damage.

In [ ]:

POP_OUT_DIR = OUT_DIR / "population"
POP_OUT_DIR.mkdir(parents=True, exist_ok=True)

if not WORLDPOP_RASTER.exists():
    raise FileNotFoundError(f"WorldPop raster not found: {WORLDPOP_RASTER}")

with rasterio.open(WORLDPOP_RASTER) as pop_src:
    pop_arr = pop_src.read(1).astype("float32")
    pop_meta = pop_src.meta.copy()
    pop_nodata = pop_src.nodata

# ---- Reproject WorldPop to JRC grid (NOTE: not perfectly mass-conserving in EPSG:4326) ----
# For stakeholder exploration, this is acceptable as a *proxy*, but we add a sanity check.
# Using bilinear for better visual quality in dashboard
POP_RESAMPLING = Resampling.bilinear

pop_reproj = np.full(dst_shape, np.nan, dtype="float32")
reproject(
    source=pop_arr,
    destination=pop_reproj,
    src_transform=pop_meta["transform"],
    src_crs=pop_meta.get("crs", "EPSG:4326"),
    dst_transform=depth0_meta["transform"],
    dst_crs=depth0_meta.get("crs", "EPSG:4326"),
    resampling=POP_RESAMPLING,
    src_nodata=pop_nodata,
    dst_nodata=np.nan
)

# ADM3 polygons intersecting AOI
adm3_aoi = adm3_gdf[adm3_gdf.intersects(AOI_BOUNDARY)].copy()
print("ADM3 intersecting AOI:", len(adm3_aoi))

from rasterio.features import geometry_mask

def polygon_mask(poly, meta) -> np.ndarray:
    """Boolean mask where True means *inside* the polygon."""
    return geometry_mask([poly], out_shape=(meta["height"], meta["width"]),
                         transform=meta["transform"], invert=True)

poly_masks = {str(row["adm3_id"]): polygon_mask(row.geometry, depth0_meta) for _, row in adm3_aoi.iterrows()}

# ---- Sanity check: compare total pop inside AOI before/after reprojection ----
def sum_worldpop_in_geom(pop_path: Path, geom) -> float:
    with rasterio.open(pop_path) as src:
        out_image, _ = rio_mask(src, [geom], crop=True, filled=False)
        a = out_image[0].astype("float32")
        nod = src.nodata
        if nod is not None:
            a = np.where(a == nod, np.nan, a)
        return float(np.nansum(a))

try:
    total_native = sum_worldpop_in_geom(WORLDPOP_RASTER, AOI_BOUNDARY)
    aoi_mask_jrc = polygon_mask(AOI_BOUNDARY, depth0_meta)
    total_reproj = float(np.nansum(np.where(aoi_mask_jrc, pop_reproj, np.nan)))
    if np.isfinite(total_native) and np.isfinite(total_reproj) and total_native > 0:
        ratio = total_reproj / total_native
        print(f"Population reprojection sanity: native={total_native:,.0f}  reprojected={total_reproj:,.0f}  ratio={ratio:.3f}")
        if abs(ratio - 1.0) > 0.10:
            print("⚠️  Warning: reprojection changes total population in AOI by >10%. Treat exposure numbers as approximate.")
except Exception as e:
    print("⚠️ Population sanity check skipped (masking failed):", e)

thr_grid = np.round(np.arange(DASHBOARD_THRESHOLD_MIN, DASHBOARD_THRESHOLD_MAX + 1e-9, DASHBOARD_THRESHOLD_STEP), 2).tolist()

affected_rows = []
for _, row in depth_df.iterrows():
    ep_id = row["episode_id"]
    rp_label = row["rp_label"]
    scenario = row["scenario"]

    depth_path = Path(row["depth_tif"])
    depth_arr, depth_meta_pop = read_raster(depth_path)
    depth_arr = depth_arr.astype("float32")
    if depth_arr.shape != dst_shape:
        depth_arr = reproject_to_match(depth_arr, depth_meta_pop, depth0_meta, dst_shape)

    for thr in thr_grid:
        flooded = np.isfinite(depth_arr) & (depth_arr >= thr)
        flooded_pop = np.where(flooded, pop_reproj, 0.0)

        for _, adm in adm3_aoi.iterrows():
            mid = str(adm["adm3_id"])
            mname = adm["adm3_name"]
            val = float(np.nansum(flooded_pop[poly_masks[mid]]))
            affected_rows.append({
                "episode_id": ep_id,
                "rp_label": rp_label,
                "scenario": scenario,
                "depth_threshold_m": float(thr),
                "adm3_id": mid,
                "adm3_name": mname,
                "affected_pop": val
            })

# ---- Compute GFM (observed) affected population for comparison ----
gfm_pop_rows = []
print("\n📊 Computing GFM-based affected population...")
for _, ep in episode_df.iterrows():
    ep_id = ep["episode_id"]
    obs_bin, valid_mask = obs_reproj_cache[ep_id]
    
    # GFM extent (binary: 0 or 1)
    flooded_gfm = (obs_bin == 1)
    flooded_gfm_pop = np.where(flooded_gfm, pop_reproj, 0.0)
    
    for _, adm in adm3_aoi.iterrows():
        mid = str(adm["adm3_id"])
        mname = adm["adm3_name"]
        val_gfm = float(np.nansum(flooded_gfm_pop[poly_masks[mid]]))
        gfm_pop_rows.append({
            "episode_id": ep_id,
            "adm3_id": mid,
            "adm3_name": mname,
            "observed_pop": val_gfm
        })

gfm_df = pd.DataFrame(gfm_pop_rows)
# Merge GFM data into affected_df (observed_pop column)
affected_df = pd.DataFrame(affected_rows)
affected_df = affected_df.merge(
    gfm_df,
    on=["episode_id", "adm3_id", "adm3_name"],
    how="left"
)
affected_csv = POP_OUT_DIR / "affected_population_by_adm3.csv"
affected_df.to_csv(affected_csv, index=False)
print("Saved:", affected_csv)
print(f"✓ Added observed_pop column (GFM-based) to {len(affected_df)} rows")
display(affected_df.head())


## 10) Build standalone HTML dashboard (Leaflet + Plotly)

Generates an interactive HTML dashboard designed for non-technical stakeholders.

**What it shows**:
- **Interactive map**: Satellite-observed (GFM, red) vs. modelled depth (blue) flood extent
- **Flood event selector**: Dropdown with human-readable event labels (customisable via `EVENT_LABELS`)
- **Municipality filter**: Click map or use dropdown to zoom into a specific area
- **Depth threshold slider**: Adjust the minimum depth shown; recommended threshold is marked
- **People exposed KPIs**: Forecast vs. satellite side-by-side, for AOI and selected municipality
- **Summary sentence**: Plain-language description of the currently selected view
- **Top 10 bar chart**: Modelled vs. observed population exposure by municipality
- **Accuracy metrics**: IoU, F1, Precision, Coverage — with plain-language explanations
- **Base map toggle**: Street / Topographic / Satellite

**Design decisions**:
- FLOPROS flood-protection scenario is **not** shown in the UI (see note in Cell 10). The dashboard
  always displays the "No Protection" baseline, which is the most interpretable view for stakeholders
  unfamiliar with flood-protection modelling.
- Accuracy metrics are displayed without traffic-light thresholds, with a caveat note explaining
  that both the model and satellite data have limitations — imperfect scores do not imply the model
  is wrong.

**Output**: `validation_dashboard.html` — fully self-contained; share by email or upload to a web server.

**Duration**: ~2–5 minutes to generate.

**Note**: Requires an internet connection for Leaflet/Plotly CDN libraries and basemap tiles.

In [ ]:
from io import BytesIO
from PIL import Image

# ─────────────────────────────────────────────────────────────────────────────
# Dashboard-only settings
# ─────────────────────────────────────────────────────────────────────────────
DASHBOARD_SCENARIO = "NoProt"  # Dashboard always shows No Protection baseline

# ─────────────────────────────────────────────────────────────────────────────
# Raster rendering helpers
# ─────────────────────────────────────────────────────────────────────────────
def raster_bounds_from_meta(meta: dict) -> Tuple[float, float, float, float]:
    t = meta["transform"]
    w, h = meta["width"], meta["height"]
    west, north = t.c, t.f
    east  = west  + t.a * w
    south = north + t.e * h
    return (min(west, east), min(south, north), max(west, east), max(south, north))


def render_extent_png(extent_bin: np.ndarray, valid_mask: np.ndarray,
                      color=(227, 24, 55), alpha=170) -> str:
    # [CHANGE] Default colour updated from pure red (255,0,0) to START Network red (227,24,55)
    h, w = extent_bin.shape
    rgba = np.zeros((h, w, 4), dtype=np.uint8)
    flood = (extent_bin == 1) & valid_mask
    rgba[flood, 0] = color[0]; rgba[flood, 1] = color[1]; rgba[flood, 2] = color[2]
    rgba[flood, 3] = alpha
    img = Image.fromarray(rgba, mode="RGBA")
    buf = BytesIO(); img.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode("utf-8")


def render_depth_png(depth: np.ndarray, vmin: float, vmax: float,
                     threshold: Optional[float] = None) -> str:
    # [CHANGE] Colormap changed from viridis to Blues — blue hues intuitively
    #          represent water depth and avoid confusion with the red observed extent.
    arr = depth.astype("float32")
    mask = np.isfinite(arr) & (arr > 0)
    if threshold is not None:
        mask = mask & (arr >= float(threshold))
    if vmax <= vmin:
        vmax = vmin + 1.0
    norm = (np.clip(arr, vmin, vmax) - vmin) / (vmax - vmin)
    cmap = plt.get_cmap("Blues")
    rgba = (cmap(np.nan_to_num(norm, nan=0.0)) * 255).astype(np.uint8)
    rgba[..., 3] = np.where(mask, 210, 0).astype(np.uint8)
    img = Image.fromarray(rgba, mode="RGBA")
    buf = BytesIO(); img.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode("utf-8")


def depth_bar_base64(width: int = 240, height: int = 12) -> str:
    # [CHANGE] Renamed from viridis_bar_base64; now uses Blues colormap to match render_depth_png
    cmap = plt.get_cmap("Blues")
    grad = np.linspace(0, 1, width)
    rgba = (cmap(grad) * 255).astype(np.uint8)
    rgba2 = np.tile(rgba[None, :, :], (height, 1, 1))
    img = Image.fromarray(rgba2, mode="RGBA")
    buf = BytesIO(); img.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode("utf-8")


def render_population_png(pop: np.ndarray, valid_mask: np.ndarray) -> str:
    """Render population density as YlOrBr heatmap (unchanged from original)."""
    arr = pop.astype("float32")
    mask = np.isfinite(arr) & (arr > 0) & valid_mask
    if mask.any():
        pmin = np.nanpercentile(arr[mask], 2)
        pmax = np.nanpercentile(arr[mask], 98)
    else:
        pmin, pmax = 0, 1
    if pmax <= pmin:
        pmax = pmin + 1
    norm = np.clip((arr - pmin) / (pmax - pmin), 0, 1)
    cmap = plt.get_cmap("YlOrBr")
    rgba = (cmap(norm) * 255).astype(np.uint8)
    rgba[..., 3] = np.where(mask, 110, 0).astype(np.uint8)
    img = Image.fromarray(rgba, mode="RGBA")
    max_dim = max(img.size)
    if max_dim > 8000:
        scale = 8000 / max_dim
        img = img.resize((int(img.size[0] * scale), int(img.size[1] * scale)), Image.Resampling.LANCZOS)
    buf = BytesIO(); img.save(buf, format="PNG", optimize=True)
    return base64.b64encode(buf.getvalue()).decode("utf-8")


depth_bar_b64 = depth_bar_base64()
pop_overlay   = render_population_png(pop_reproj, np.isfinite(pop_reproj))
print("✓ Population overlay rendered")

dashboard_rp_label = "envelope"

# ─────────────────────────────────────────────────────────────────────────────
# Build observed overlays (GFM satellite extent)
# ─────────────────────────────────────────────────────────────────────────────
obs_overlays = {}
for ep_id, (obs_bin, valid_mask) in obs_reproj_cache.items():
    obs_overlays[ep_id] = render_extent_png(obs_bin, valid_mask, color=(227, 24, 55), alpha=170)

# ─────────────────────────────────────────────────────────────────────────────
# Build depth overlays — NoProt scenario only
# [CHANGE] Removed scenario dimension: depth_overlays[ep_id][thr_key]
#          (was: depth_overlays[ep_id][scenario][thr_key])
# [BUG FIX] Filtering to DASHBOARD_SCENARIO prevents a silent "no data" state
#           when the scenario toggle was removed.
# ─────────────────────────────────────────────────────────────────────────────
if "thr_grid" not in globals():
    thr_grid = [0.0, 0.2, 0.5, 1.0, 2.0, 3.0]

depth_overlays = {}
depth_ranges   = {}

for _, row in depth_df.iterrows():
    if row["rp_label"] != dashboard_rp_label:
        continue
    if row["scenario"] != DASHBOARD_SCENARIO:
        continue
    ep_id      = row["episode_id"]
    depth_path = Path(row["depth_tif"])
    depth_arr, _ = read_raster(depth_path)
    depth_arr = depth_arr.astype("float32")
    vmin = float(np.nanpercentile(depth_arr, 1))  if np.isfinite(depth_arr).any() else 0.0
    vmax = float(np.nanpercentile(depth_arr, 99)) if np.isfinite(depth_arr).any() else 1.0
    if not np.isfinite(vmin): vmin = 0.0
    if not np.isfinite(vmax) or vmax <= vmin: vmax = vmin + 1.0
    depth_ranges[ep_id] = {"vmin": vmin, "vmax": vmax}  # No scenario nesting
    for thr in thr_grid:
        thr_key = f"{thr:.2f}"
        depth_overlays.setdefault(ep_id, {})[thr_key] = render_depth_png(
            depth_arr, vmin=vmin, vmax=vmax, threshold=float(thr)
        )

west, south, east, north = raster_bounds_from_meta(depth0_meta)
leaflet_bounds = [[south, west], [north, east]]
adm3_geojson   = json.loads(adm3_aoi.to_json())

# ─────────────────────────────────────────────────────────────────────────────
# Helper: normalise adm3_id to string robustly
# [BUG FIX] Handles int, numpy int64, and float representations uniformly so
#           JS String(feature.properties.adm3_id) and Python dict keys always match.
# ─────────────────────────────────────────────────────────────────────────────
def _norm_id(x) -> str:
    try:
        return str(int(float(x)))
    except Exception:
        return str(x)

# ─────────────────────────────────────────────────────────────────────────────
# Compact data structures — NoProt only, scenario dimension removed
# [CHANGE] All data_compact / totals_compact / metrics_compact are now keyed as
#          [ep_id][thr] instead of [ep_id][scenario][thr].
# ─────────────────────────────────────────────────────────────────────────────
aff_noprot = affected_df[
    (affected_df["rp_label"] == dashboard_rp_label) &
    (affected_df["scenario"] == DASHBOARD_SCENARIO)
].copy()

data_compact = {}
for ep_id, grp in aff_noprot.groupby("episode_id"):
    dd = {}
    for thr, g2 in grp.groupby("depth_threshold_m"):
        dd[f"{thr:.2f}"] = {_norm_id(r["adm3_id"]): float(r["affected_pop"]) for _, r in g2.iterrows()}
    data_compact[ep_id] = dd

obs_data_compact = {}
for ep_id, grp in aff_noprot.groupby("episode_id"):
    obs_data_compact[ep_id] = {
        _norm_id(r["adm3_id"]): float(r["observed_pop"]) if pd.notna(r["observed_pop"]) else 0.0
        for _, r in grp.drop_duplicates(subset=["episode_id", "adm3_id"]).iterrows()
    }

totals_compact = {}
for ep_id, grp in aff_noprot.groupby("episode_id"):
    totals_compact[ep_id] = {
        f"{thr:.2f}": float(g2["affected_pop"].sum())
        for thr, g2 in grp.groupby("depth_threshold_m")
    }

obs_totals_compact = {}
for ep_id, grp in aff_noprot.groupby("episode_id"):
    obs_totals_compact[ep_id] = float(
        grp.drop_duplicates(subset=["episode_id", "adm3_id"])["observed_pop"].fillna(0.0).sum()
    )

metrics_compact = {}
m_filt = metrics_df[
    (metrics_df["rp_label"] == dashboard_rp_label) &
    (metrics_df["scenario"] == DASHBOARD_SCENARIO)
].copy()
for ep_id, grp in m_filt.groupby("episode_id"):
    metrics_compact[ep_id] = {
        f"{r['depth_threshold_m']:.2f}": {
            "iou":       float(r["iou"])       if pd.notna(r["iou"])       else None,
            "f1":        float(r["f1"])        if pd.notna(r["f1"])        else None,
            "precision": float(r["precision"]) if pd.notna(r["precision"]) else None,
            "recall":    float(r["recall"])    if pd.notna(r["recall"])    else None,
        } for _, r in grp.iterrows()
    }

# ─────────────────────────────────────────────────────────────────────────────
# Recommended threshold per episode (highest F1)
# Used to mark the slider in the dashboard UI
# ─────────────────────────────────────────────────────────────────────────────
rec_thresholds = {}
for ep_id, grp in m_filt.groupby("episode_id"):
    try:
        valid = grp[grp["f1"].notna()]
        if len(valid) > 0:
            best_idx = valid["f1"].idxmax()
            rec_thresholds[ep_id] = f"{valid.loc[best_idx, 'depth_threshold_m']:.2f}"
        else:
            rec_thresholds[ep_id] = "0.20"
    except Exception:
        rec_thresholds[ep_id] = "0.20"

# ─────────────────────────────────────────────────────────────────────────────
# Event labels — custom from EVENT_LABELS dict, auto-generated fallback
# [NEW] Users set EVENT_LABELS in Cell 10 (User Controls)
# ─────────────────────────────────────────────────────────────────────────────
event_labels_js = {}
for ep_id in sorted(data_compact.keys()):
    if ep_id in EVENT_LABELS and EVENT_LABELS[ep_id]:
        event_labels_js[ep_id] = EVENT_LABELS[ep_id]
    else:
        try:
            parts   = ep_id.split("_")
            ep_num  = int(parts[0].replace("EP", ""))
            start_dt = pd.to_datetime(parts[1], format="%Y%m%d")
            end_dt   = pd.to_datetime(parts[2], format="%Y%m%d")
            if start_dt.year == end_dt.year and start_dt.month == end_dt.month:
                label = f"Event {ep_num} ({start_dt.strftime('%b %d')}–{end_dt.strftime('%d, %Y')})"
            elif start_dt.year == end_dt.year:
                label = f"Event {ep_num} ({start_dt.strftime('%b %d')}–{end_dt.strftime('%b %d, %Y')})"
            else:
                label = f"Event {ep_num} ({start_dt.strftime('%b %d, %Y')}–{end_dt.strftime('%b %d, %Y')})"
        except Exception:
            label = ep_id
        event_labels_js[ep_id] = label

thr_grid_js = [f"{t:.2f}" for t in thr_grid]

print(f"✅ Dashboard data prepared:")
print(f"   Flood events : {len(data_compact)}")
print(f"   Event labels : {event_labels_js}")
print(f"   Rec. thresholds: {rec_thresholds}")

# ─────────────────────────────────────────────────────────────────────────────
# HTML DASHBOARD
# ─────────────────────────────────────────────────────────────────────────────
html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8"/>
<title>Flood Forecast Validation — {BASIN_DISPLAY_NAME}</title>
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"/>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<script src="https://cdn.plot.ly/plotly-2.30.0.min.js"></script>
<link href="https://fonts.googleapis.com/css2?family=DM+Serif+Display&family=IBM+Plex+Sans:wght@400;500;600&family=IBM+Plex+Mono:wght@500&display=swap" rel="stylesheet">
<style>
:root {{
  --red:        #E31837;
  --navy:       #0F2044;
  --blue:       #2563EB;
  --blue-lt:    #EFF6FF;
  --amber:      #D97706;
  --ink:        #1A202C;
  --muted:      #64748B;
  --border:     #E2E8F0;
  --border-md:  #CBD5E0;
  --surface:    #FFFFFF;
  --surface-alt:#F8FAFC;
  --indigo-lt:  #EEF2FF;
  --indigo:     #6366F1;
  --indigo-dk:  #3730A3;
  --indigo-xdk: #1E1B4B;
}}
* {{ box-sizing: border-box; margin: 0; padding: 0; }}
html, body {{ height: 100%; font-family: 'IBM Plex Sans', sans-serif; color: var(--ink); background: var(--surface-alt); }}

/* Layout */
#shell {{ display: flex; flex-direction: column; height: 100vh; }}
#topbar {{
  background: var(--navy);
  color: #fff;
  padding: 0 18px;
  height: 50px;
  display: flex;
  align-items: center;
  gap: 10px;
  border-bottom: 3px solid var(--red);
  flex-shrink: 0;
}}
.topbar-org {{ font-size: 10px; font-weight: 600; letter-spacing: 0.1em; text-transform: uppercase; color: rgba(255,255,255,0.45); }}
.topbar-sep {{ color: var(--red); font-size: 16px; line-height: 1; }}
.topbar-title {{ font-family: 'DM Serif Display', serif; font-size: 16px; color: #fff; flex: 1; }}
.topbar-meta {{ font-size: 10px; color: rgba(255,255,255,0.4); }}
#main {{ display: flex; flex: 1; overflow: hidden; }}
#sidebar {{ width: 360px; overflow-y: auto; border-right: 1px solid var(--border); background: var(--surface); flex-shrink: 0; }}
#mapWrap {{ position: relative; flex: 1; }}
#map {{ height: 100%; width: 100%; }}

/* Sections */
.section {{ padding: 13px 15px; border-bottom: 1px solid var(--border); }}
.sec-title {{ font-size: 9px; font-weight: 600; letter-spacing: 0.1em; text-transform: uppercase; color: var(--muted); margin-bottom: 9px; }}

/* Form controls */
.field {{ margin-bottom: 9px; }}
.field:last-child {{ margin-bottom: 0; }}
.field-label {{ font-size: 11px; font-weight: 600; color: var(--ink); display: flex; align-items: center; gap: 5px; margin-bottom: 4px; }}
.tip {{ cursor: help; color: var(--muted); font-size: 11px; }}
select {{
  width: 100%; padding: 6px 9px; border: 1px solid var(--border-md);
  border-radius: 6px; font-family: inherit; font-size: 13px;
  background: var(--surface); color: var(--ink); cursor: pointer;
}}
select:focus {{ outline: 2px solid var(--blue); border-color: transparent; }}

/* Slider */
.slider-wrap {{ padding-bottom: 18px; position: relative; }}
input[type=range] {{ width: 100%; accent-color: var(--navy); cursor: pointer; display: block; }}
.slider-scale {{ display: flex; justify-content: space-between; font-size: 10px; color: var(--muted); margin-top: 3px; }}
.slider-current {{ text-align: center; font-family: 'IBM Plex Mono', monospace; font-size: 14px; font-weight: 500; color: var(--navy); margin-top: 4px; }}
.rec-pin {{
  position: absolute; bottom: 0; font-size: 9px; color: var(--amber);
  white-space: nowrap; transform: translateX(-50%); pointer-events: none; display: none;
}}
.rec-pin::before {{ content: "▲"; display: block; text-align: center; line-height: 1; }}

/* KPI cards */
.kpi-row {{ display: grid; grid-template-columns: 1fr 1fr; gap: 8px; }}
.kpi-card {{
  border: 1px solid var(--border); border-radius: 8px; padding: 10px 12px;
  background: var(--surface); position: relative; overflow: hidden;
}}
.kpi-card::after {{ content: ''; position: absolute; top: 0; left: 0; right: 0; height: 3px; }}
.kpi-forecast::after {{ background: var(--blue); }}
.kpi-observed::after {{ background: var(--red); }}
.kpi-lbl {{ font-size: 9px; font-weight: 600; letter-spacing: 0.08em; text-transform: uppercase; color: var(--muted); margin-bottom: 3px; }}
.kpi-num {{ font-family: 'IBM Plex Mono', monospace; font-size: 20px; font-weight: 500; line-height: 1.1; }}
.kpi-forecast .kpi-num {{ color: var(--blue); }}
.kpi-observed .kpi-num {{ color: var(--red); }}
.kpi-sub {{ font-size: 10px; color: var(--muted); margin-top: 2px; }}
.muni-card {{
  margin-top: 8px; display: none; border: 1px solid var(--border);
  border-radius: 8px; padding: 9px 12px; background: var(--surface-alt);
}}
.muni-card .muni-name {{ font-size: 12px; font-weight: 600; margin-bottom: 5px; color: var(--navy); }}
.muni-row {{ display: flex; justify-content: space-between; align-items: baseline; font-size: 12px; margin-bottom: 2px; }}
.muni-row .ml {{ color: var(--muted); }}
.muni-row .mv {{ font-family: 'IBM Plex Mono', monospace; font-weight: 500; }}

/* Summary */
.summary {{
  background: #F0F4FF; border-left: 3px solid var(--navy);
  border-radius: 0 6px 6px 0; padding: 8px 11px;
  font-size: 12px; line-height: 1.6; color: var(--ink);
}}

/* Charts & metrics */
.chart-note {{ font-size: 10px; color: var(--muted); margin-top: 5px; font-style: italic; line-height: 1.4; }}
.metric-grid {{ display: grid; grid-template-columns: 1fr 1fr; gap: 6px; margin-bottom: 9px; }}
.metric-card {{
  border: 1px solid var(--border); border-radius: 6px;
  padding: 8px 10px; background: var(--surface);
}}
.metric-name {{ font-size: 10px; font-weight: 600; color: var(--muted); display: flex; align-items: center; gap: 4px; margin-bottom: 2px; }}
.metric-val {{ font-family: 'IBM Plex Mono', monospace; font-size: 19px; color: var(--navy); font-weight: 500; }}
.metric-desc {{ font-size: 10px; color: var(--muted); margin-top: 3px; line-height: 1.4; }}
.caveat-box {{
  font-size: 11px; color: #92400E; line-height: 1.5;
  background: #FFFBEB; border: 1px solid #FDE68A; border-radius: 6px; padding: 8px 10px;
}}

/* Map overlays */
.map-status {{
  position: absolute; top: 11px; left: 11px; z-index: 900;
  background: rgba(255,255,255,0.95); border: 1px solid var(--border);
  border-radius: 8px; padding: 7px 11px; font-size: 12px;
  box-shadow: 0 4px 12px rgba(0,0,0,0.08); max-width: 260px; line-height: 1.5;
}}
.map-legend {{
  position: absolute; right: 11px; bottom: 28px; z-index: 900;
  background: rgba(255,255,255,0.96); border: 1px solid var(--border);
  border-radius: 10px; padding: 10px 12px; font-size: 11px;
  box-shadow: 0 4px 12px rgba(0,0,0,0.08); min-width: 190px;
}}
.leg-title {{ font-size: 9px; font-weight: 600; text-transform: uppercase; letter-spacing: 0.08em; color: var(--muted); margin-bottom: 5px; }}
.leg-row {{ display: flex; align-items: center; gap: 7px; margin-bottom: 3px; }}
.swatch {{ width: 13px; height: 13px; border-radius: 3px; flex-shrink: 0; }}
.ramp-wrap {{ margin-top: 6px; }}
.ramp-img {{ width: 100%; height: 9px; border-radius: 3px; border: 1px solid var(--border); display: block; margin: 2px 0; }}
.ramp-labels {{ display: flex; justify-content: space-between; font-size: 10px; color: var(--muted); }}

/* Layer control */
.lbtn {{
  background: var(--surface); border: 1px solid var(--border);
  border-radius: 6px; padding: 6px 10px; font-family: inherit;
  font-size: 12px; cursor: pointer; box-shadow: 0 2px 5px rgba(0,0,0,0.07);
  white-space: nowrap;
}}
.lbtn:hover {{ background: var(--surface-alt); }}
.lpanel {{
  display: none; background: var(--surface); border: 1px solid var(--border);
  border-radius: 8px; padding: 10px 12px; margin-top: 4px;
  font-size: 12px; box-shadow: 0 4px 12px rgba(0,0,0,0.08); min-width: 200px;
}}
.lpanel.open {{ display: block; }}
.lgroup {{ font-size: 9px; font-weight: 600; text-transform: uppercase; letter-spacing: 0.08em; color: var(--muted); margin: 8px 0 4px; }}
.lgroup:first-child {{ margin-top: 0; }}
.lrow {{ display: flex; align-items: center; gap: 7px; margin: 3px 0; line-height: 1.4; }}
.lrow label {{ cursor: pointer; }}
.reset-btn {{
  width: 100%; margin-top: 8px; padding: 5px;
  background: var(--surface-alt); border: 1px solid var(--border);
  border-radius: 5px; font-family: inherit; font-size: 11px; cursor: pointer;
}}

@media (max-width: 900px) {{
  #main {{ flex-direction: column; }}
  #sidebar {{ width: 100%; height: 52vh; border-right: none; border-bottom: 1px solid var(--border); }}
  #mapWrap {{ height: 48vh; }}
}}
</style>
</head>
<body>
<div id="shell">

  <!-- Top bar -->
  <div id="topbar">
    <span class="topbar-org">START Network</span>
    <span class="topbar-sep">·</span>
    <span class="topbar-title">Flood Forecast Validation — {BASIN_DISPLAY_NAME}</span>
    <span class="topbar-meta">Philippines · GloFAS–JRC Pipeline</span>
  </div>

  <div id="main">
    <!-- Sidebar -->
    <div id="sidebar">

      <!-- Event + Municipality selectors -->
      <div class="section">
        <div class="sec-title">Event Selection</div>
        <div class="field">
          <div class="field-label">Flood Event</div>
          <select id="eventSelect"></select>
        </div>
        <div class="field">
          <div class="field-label">
            Municipality
            <span class="tip" title="Filter population data to one municipality, or leave at 'All' for the full basin.">ⓘ</span>
          </div>
          <select id="muniSelect"></select>
        </div>
      </div>

      <!-- Depth slider -->
      <div class="section">
        <div class="sec-title">Display Filter</div>
        <div class="field">
          <div class="field-label">
            Minimum Flood Depth
            <span class="tip" title="Show only areas where the model predicts water depth above this value. Lower = larger area shown; higher = only deeper floods.">ⓘ</span>
          </div>
          <div class="slider-wrap">
            <input type="range" id="thrSlider"
              min="{DASHBOARD_THRESHOLD_MIN}" max="{DASHBOARD_THRESHOLD_MAX}"
              step="{DASHBOARD_THRESHOLD_STEP}" value="0.20"/>
            <div class="slider-scale">
              <span>{DASHBOARD_THRESHOLD_MIN} m</span>
              <span>{DASHBOARD_THRESHOLD_MAX} m</span>
            </div>
            <div class="slider-current">Showing depth ≥ <b><span id="thrValue">0.20</span> m</b></div>
            <div class="rec-pin" id="recPin">Best match</div>
          </div>
        </div>
      </div>

      <!-- KPI cards -->
      <div class="section">
        <div class="sec-title">People Exposed</div>
        <div class="kpi-row">
          <div class="kpi-card kpi-forecast">
            <div class="kpi-lbl">Forecast</div>
            <div class="kpi-num" id="kpiTotal">—</div>
            <div class="kpi-sub">people at risk</div>
          </div>
          <div class="kpi-card kpi-observed">
            <div class="kpi-lbl">Satellite</div>
            <div class="kpi-num" id="kpiObsTotal">—</div>
            <div class="kpi-sub">observed flood</div>
          </div>
        </div>
        <div class="muni-card" id="muniCard">
          <div class="muni-name" id="muniCardName"></div>
          <div class="muni-row">
            <span class="ml">Forecast</span>
            <span class="mv" id="kpiMuni">—</span>
          </div>
          <div class="muni-row">
            <span class="ml">Satellite</span>
            <span class="mv" id="kpiMuniObs">—</span>
          </div>
        </div>
      </div>

      <!-- Summary sentence -->
      <div class="section">
        <div class="summary" id="summaryBox">Loading…</div>
      </div>

      <!-- Bar chart -->
      <div class="section">
        <div class="sec-title">Top 10 Municipalities by Exposure</div>
        <div id="barChart" style="height:260px;"></div>
        <div class="chart-note">
          Blue = Forecast · Red = Satellite observed.
          Numbers are <em>people exposed</em>, not damage counts.
        </div>
      </div>

      <!-- Accuracy metrics -->
      <div class="section">
        <div class="sec-title">Forecast Accuracy</div>
        <div class="metric-grid">
          <div class="metric-card">
            <div class="metric-name">
              Map Match Score
              <span class="tip" title="Intersection over Union (IoU): overlap between forecast and satellite areas divided by their combined area. 0 = no overlap, 1 = perfect.">ⓘ</span>
            </div>
            <div class="metric-val" id="metIou">—</div>
            <div class="metric-desc">How much the forecast and satellite flood areas overlap</div>
          </div>
          <div class="metric-card">
            <div class="metric-name">
              F1 Score
              <span class="tip" title="Harmonic mean of Precision and Coverage. Balances over- and under-prediction. Range 0–1.">ⓘ</span>
            </div>
            <div class="metric-val" id="metF1">—</div>
            <div class="metric-desc">Balance between false alarms and missed flooding</div>
          </div>
          <div class="metric-card">
            <div class="metric-name">
              Precision
              <span class="tip" title="Of all forecast flood areas, what fraction actually flooded? High = fewer false alarms.">ⓘ</span>
            </div>
            <div class="metric-val" id="metPrec">—</div>
            <div class="metric-desc">Fraction of forecast area confirmed by satellite</div>
          </div>
          <div class="metric-card">
            <div class="metric-name">
              Coverage
              <span class="tip" title="Recall: of all satellite-observed flood areas, what fraction did the model capture? High = fewer missed areas.">ⓘ</span>
            </div>
            <div class="metric-val" id="metRec">—</div>
            <div class="metric-desc">Fraction of satellite flood area captured by forecast</div>
          </div>
        </div>
        <div class="caveat-box">
          ⚠️ Both the forecast model and satellite data have known limitations (cloud cover,
          timing differences, mapping uncertainty). Scores below 1.0 are expected and do not
          indicate that the model is incorrect.
        </div>
      </div>

    </div><!-- /sidebar -->

    <!-- Map -->
    <div id="mapWrap">
      <div id="map"></div>
      <div class="map-status" id="mapStatus">Loading map…</div>
      <div class="map-legend">
        <div class="leg-title">Legend</div>
        <div class="leg-row"><div class="swatch" style="background:rgba(227,24,55,0.72);"></div><span>Satellite extent (GFM)</span></div>
        <div class="leg-row"><div class="swatch" style="background:rgba(37,99,235,0.72);"></div><span>Forecast depth (model)</span></div>
        <div class="leg-row"><div class="swatch" style="background:rgba(217,119,6,0.45);"></div><span>Population density</span></div>
        <div class="ramp-wrap" id="depthRampWrap">
          <div class="leg-title" style="margin-top:7px;">Forecast Depth (m)</div>
          <img class="ramp-img" id="depthRamp" alt="depth scale"/>
          <div class="ramp-labels">
            <span id="depthMinLbl">—</span>
            <span id="depthMaxLbl">—</span>
          </div>
        </div>
        <div id="choroLegend"></div>
      </div>
    </div>

  </div><!-- /main -->
</div><!-- /shell -->

<script>
// ── Data from Python ──────────────────────────────────────────────────────
const BOUNDS        = {json.dumps(leaflet_bounds)};
const OBS_OVERLAYS  = {json.dumps(obs_overlays)};
const POP_OVERLAY   = "{pop_overlay}";
const DEPTH_OVERLAYS= {json.dumps(depth_overlays)};
const DEPTH_RANGES  = {json.dumps(depth_ranges)};
const DEPTH_BAR     = "data:image/png;base64,{depth_bar_b64}";
const ADM3_GEOJSON  = {json.dumps(adm3_geojson)};
const DATA          = {json.dumps(data_compact)};
const OBS_DATA      = {json.dumps(obs_data_compact)};
const TOTALS        = {json.dumps(totals_compact)};
const OBS_TOTALS    = {json.dumps(obs_totals_compact)};
const METRICS       = {json.dumps(metrics_compact)};
const REC_THR       = {json.dumps(rec_thresholds)};
const EVT_LABELS    = {json.dumps(event_labels_js)};
const THR_GRID      = {json.dumps(thr_grid_js)};
const BASIN         = "{BASIN_DISPLAY_NAME}";

// ── State ─────────────────────────────────────────────────────────────────
let layerVis = {{ obs:true, depth:true, pop:false, muni:true }};
let currentBase = "osm";
let obsLyr=null, depthLyr=null, popLyr=null, muniLyr=null;
let selMuni = null, updateTimer=null, lastKey="";

// ── Map init ──────────────────────────────────────────────────────────────
const map = L.map('map', {{zoomControl:true}}).fitBounds(BOUNDS);
['obsPane','depthPane','popPane','muniPane'].forEach((p,i) => {{
  map.createPane(p); map.getPane(p).style.zIndex = 410 + i*10;
}});
const baseLayers = {{
  osm:  L.tileLayer('https://{{s}}.tile.openstreetmap.org/{{z}}/{{x}}/{{y}}.png',
          {{maxZoom:13, attribution:'© OpenStreetMap'}}),
  topo: L.tileLayer('https://{{s}}.tile.opentopomap.org/{{z}}/{{x}}/{{y}}.png',
          {{maxZoom:13, attribution:'© OpenTopoMap'}}),
  sat:  L.tileLayer('https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{{z}}/{{y}}/{{x}}',
          {{maxZoom:13, attribution:'Tiles © Esri'}})
}};
let baseLayer = baseLayers.osm.addTo(map);

// ── Helpers ───────────────────────────────────────────────────────────────
function nearestThr(v) {{
  let best=THR_GRID[0], bd=Math.abs(v-parseFloat(best));
  for (const t of THR_GRID) {{
    const d=Math.abs(v-parseFloat(t));
    if (d<bd) {{ bd=d; best=t; }}
  }}
  return parseFloat(best).toFixed(2);
}}
function fmt(x) {{
  if (x==null || !isFinite(x)) return "—";
  return Math.round(x).toLocaleString();
}}
function fmtD(x) {{
  if (x==null || !isFinite(x) || x==="") return "—";
  return Number(x).toFixed(2);
}}
function evtLabel(id) {{ return EVT_LABELS[id] || id; }}
function breaks(vals) {{
  const v=Object.values(vals).filter(x=>isFinite(x)&&x>0).sort((a,b)=>a-b);
  if (!v.length) return [0,1,2,3,4];
  const q=p=>v[Math.floor(p*(v.length-1))];
  return [0,q(.25),q(.5),q(.75),v[v.length-1]];
}}
function choroClr(val,brk) {{
  if (val==null||!isFinite(val)||val===0) return "#F1F5F9";
  if (val<=brk[1]) return "#C7D2FE";
  if (val<=brk[2]) return "#818CF8";
  if (val<=brk[3]) return "#4338CA";
  return "#1E1B4B";
}}
function setBase(k) {{
  if (baseLayer) map.removeLayer(baseLayer);
  currentBase=k;
  baseLayer=(baseLayers[k]||baseLayers.osm).addTo(map);
  syncUI();
}}
function syncUI() {{
  [['chkObs','obs'],['chkDepth','depth'],['chkPop','pop'],['chkMuni','muni']].forEach(([id,k])=>{{
    const el=document.getElementById(id); if(el) el.checked=!!layerVis[k];
  }});
  document.querySelectorAll('[name=baseMap]').forEach(r=>{{r.checked=(r.value===currentBase);}});
}}

// ── Recommended threshold pin ─────────────────────────────────────────────
function updatePin(ep) {{
  const pin=document.getElementById('recPin'), sl=document.getElementById('thrSlider');
  if (!pin||!sl||!REC_THR[ep]) {{ if(pin) pin.style.display='none'; return; }}
  const rec=parseFloat(REC_THR[ep]);
  const pct=(rec-parseFloat(sl.min))/(parseFloat(sl.max)-parseFloat(sl.min))*100;
  pin.style.left=pct+'%'; pin.style.display='block';
}}

// ── Dropdowns ─────────────────────────────────────────────────────────────
function populateEvents() {{
  const s=document.getElementById('eventSelect'); s.innerHTML='';
  Object.keys(DATA).sort().forEach(ep=>{{
    const o=document.createElement('option'); o.value=ep; o.text=evtLabel(ep); s.appendChild(o);
  }});
}}
function populateMunis() {{
  const s=document.getElementById('muniSelect');
  s.innerHTML='<option value="">All municipalities</option>';
  ADM3_GEOJSON.features.forEach(f=>{{
    const o=document.createElement('option');
    o.value=String(f.properties.adm3_id); o.text=f.properties.adm3_name; s.appendChild(o);
  }});
}}

// ── Main update ───────────────────────────────────────────────────────────
function update() {{
  const ep  = document.getElementById('eventSelect').value;
  const thr = nearestThr(parseFloat(document.getElementById('thrSlider').value));

  if (!ep||!DATA[ep]||!DATA[ep][thr]) {{
    document.getElementById('mapStatus').textContent='No data for this selection.'; return;
  }}
  if (!DEPTH_OVERLAYS[ep]||!DEPTH_OVERLAYS[ep][thr]) {{
    document.getElementById('mapStatus').textContent='Map imagery not ready for this selection.'; return;
  }}

  document.getElementById('thrValue').textContent=thr;
  updatePin(ep);

  const opEl=document.getElementById('muniOpSlider');
  const op=opEl ? parseInt(opEl.value)/100 : 0.30;
  if (document.getElementById('muniOpVal')) document.getElementById('muniOpVal').textContent=Math.round(op*100)+'%';

  const stateKey=[ep,thr,...Object.values(layerVis),op,selMuni].join('|');
  if (stateKey===lastKey) return;
  lastKey=stateKey;

  // status strip
  document.getElementById('mapStatus').innerHTML=
    '<b>'+evtLabel(ep)+'</b><br/>Showing depth ≥ <b>'+thr+' m</b>';

  // observed layer
  if (obsLyr) {{ map.removeLayer(obsLyr); obsLyr=null; }}
  if (layerVis.obs)
    obsLyr=L.imageOverlay("data:image/png;base64,"+OBS_OVERLAYS[ep], BOUNDS, {{opacity:0.75,pane:'obsPane'}}).addTo(map);

  // depth layer
  if (depthLyr) {{ map.removeLayer(depthLyr); depthLyr=null; }}
  if (layerVis.depth)
    depthLyr=L.imageOverlay("data:image/png;base64,"+DEPTH_OVERLAYS[ep][thr], BOUNDS, {{opacity:0.65,pane:'depthPane'}}).addTo(map);

  // population layer
  if (popLyr) {{ map.removeLayer(popLyr); popLyr=null; }}
  if (layerVis.pop)
    popLyr=L.imageOverlay("data:image/png;base64,"+POP_OVERLAY, BOUNDS, {{opacity:0.45,pane:'popPane'}}).addTo(map);

  // depth ramp legend
  const dr=DEPTH_RANGES[ep];
  if (dr) {{
    document.getElementById('depthRamp').src=DEPTH_BAR;
    document.getElementById('depthMinLbl').textContent=fmtD(dr.vmin)+' m';
    document.getElementById('depthMaxLbl').textContent=fmtD(dr.vmax)+' m';
    document.getElementById('depthRampWrap').style.display=layerVis.depth?'block':'none';
  }}

  // choropleth
  const vals=DATA[ep][thr];
  const obsVals=OBS_DATA[ep]||{{}};
  const brk=breaks(vals);

  if (muniLyr) {{ map.removeLayer(muniLyr); muniLyr=null; }}
  if (layerVis.muni) {{
    muniLyr=L.geoJSON(ADM3_GEOJSON, {{
      pane:'muniPane',
      style: f=>{{
        const id=String(f.properties.adm3_id);
        const isSel=selMuni&&id===selMuni;
        return {{
          color: isSel?'#0F2044':'#94A3B8', weight: isSel?2.5:0.8,
          fillColor: choroClr(vals[id], brk), fillOpacity: op,
          dashArray: isSel?'6,4':null
        }};
      }},
      onEachFeature: (f,lyr)=>{{
        const id=String(f.properties.adm3_id);
        lyr.on('click', ()=>{{
          selMuni=id;
          document.getElementById('muniSelect').value=id;
          update();
        }});
        lyr.bindTooltip(
          '<b>'+f.properties.adm3_name+'</b>'+
          '<br/>Forecast: <b>'+fmt(vals[id])+'</b> people'+
          '<br/>Satellite: <b>'+fmt(obsVals[id])+'</b> people',
          {{sticky:true}}
        );
      }}
    }}).addTo(map);
    map.invalidateSize();
  }}

  // choropleth legend
  const cc=['#C7D2FE','#818CF8','#4338CA','#1E1B4B'];
  const fn=x=>x>1000?(x/1000).toFixed(1)+'k':Math.round(x).toLocaleString();
  const lbls=['1 – '+fn(brk[1]),fn(brk[1])+' – '+fn(brk[2]),fn(brk[2])+' – '+fn(brk[3]),'> '+fn(brk[3])];
  document.getElementById('choroLegend').innerHTML=
    '<div class="leg-title" style="margin-top:7px;">Forecast Exposure</div>'+
    cc.map((c,i)=>'<div class="leg-row"><div class="swatch" style="background:'+c+';opacity:'+(op+0.4)+';"></div><span>'+lbls[i]+' people</span></div>').join('');

  // KPI cards
  const tot=TOTALS[ep]?TOTALS[ep][thr]:null;
  const obsTot=OBS_TOTALS[ep]||0;
  document.getElementById('kpiTotal').textContent=fmt(tot);
  document.getElementById('kpiObsTotal').textContent=fmt(obsTot);

  const mc=document.getElementById('muniCard');
  if (selMuni) {{
    const feat=ADM3_GEOJSON.features.find(f=>String(f.properties.adm3_id)===selMuni);
    document.getElementById('muniCardName').textContent=feat?feat.properties.adm3_name:selMuni;
    document.getElementById('kpiMuni').textContent=fmt(vals[selMuni]);
    document.getElementById('kpiMuniObs').textContent=fmt(obsVals[selMuni]||0);
    mc.style.display='block';
  }} else {{
    mc.style.display='none';
  }}

  // Summary sentence
  const areaName=selMuni
    ?(ADM3_GEOJSON.features.find(f=>String(f.properties.adm3_id)===selMuni)?.properties.adm3_name||BASIN)
    :BASIN;
  document.getElementById('summaryBox').innerHTML=
    'During <b>'+evtLabel(ep)+'</b>, the forecast estimates '+
    '<b style="color:#2563EB">'+fmt(tot)+' people</b> at risk of flooding '+
    '≥ '+thr+' m in '+areaName+' — compared to '+
    '<b style="color:#E31837">'+fmt(obsTot)+' people</b> in the satellite-observed flood extent.';

  // Accuracy metrics
  const m=METRICS[ep]?METRICS[ep][thr]:null;
  document.getElementById('metIou').textContent  =m?fmtD(m.iou)      :'—';
  document.getElementById('metF1').textContent   =m?fmtD(m.f1)       :'—';
  document.getElementById('metPrec').textContent =m?fmtD(m.precision):'—';
  document.getElementById('metRec').textContent  =m?fmtD(m.recall)   :'—';

  // Bar chart
  const entries=Object.entries(vals).map(([k,v])=>{{
    const feat=ADM3_GEOJSON.features.find(f=>String(f.properties.adm3_id)===k);
    return {{name:feat?feat.properties.adm3_name:k, mod:v||0, obs:obsVals[k]||0}};
  }}).sort((a,b)=>b.mod-a.mod).slice(0,10);

  Plotly.react('barChart',
    [
      {{type:'bar',orientation:'h',
        x:entries.map(e=>e.mod).reverse(), y:entries.map(e=>e.name).reverse(),
        name:'Forecast', marker:{{color:'rgba(37,99,235,0.75)'}},
        hovertemplate:'%{{y}}<br>Forecast: %{{x:,.0f}} people<extra></extra>'}},
      {{type:'bar',orientation:'h',
        x:entries.map(e=>e.obs).reverse(), y:entries.map(e=>e.name).reverse(),
        name:'Satellite', marker:{{color:'rgba(227,24,55,0.65)'}},
        hovertemplate:'%{{y}}<br>Satellite: %{{x:,.0f}} people<extra></extra>'}}
    ],
    {{
      barmode:'overlay',
      paper_bgcolor:'transparent', plot_bgcolor:'transparent',
      margin:{{l:120,r:10,t:6,b:35}},
      xaxis:{{title:'People exposed',automargin:true,tickformat:',.0f',
              gridcolor:'#E2E8F0',zeroline:false}},
      yaxis:{{automargin:true,tickfont:{{size:10}}}},
      legend:{{orientation:'h',x:0,y:-0.18,font:{{size:11}}}},
      font:{{family:'IBM Plex Sans, sans-serif',size:11,color:'#1A202C'}}
    }},
    {{displayModeBar:false, responsive:true}}
  );
}}

function sched() {{ if(updateTimer) clearTimeout(updateTimer); updateTimer=setTimeout(update,150); }}

// ── Layer control ─────────────────────────────────────────────────────────
const lCtrl=L.control({{position:'topright'}});
lCtrl.onAdd=function(){{
  const div=L.DomUtil.create('div');
  div.innerHTML=`
    <button class="lbtn" id="lBtn">⊞ Layers</button>
    <div class="lpanel" id="lPanel">
      <div class="lgroup">Base Map</div>
      <div class="lrow"><label><input type="radio" name="baseMap" value="osm" checked> Street</label></div>
      <div class="lrow"><label><input type="radio" name="baseMap" value="topo"> Topographic</label></div>
      <div class="lrow"><label><input type="radio" name="baseMap" value="sat"> Satellite</label></div>
      <div class="lgroup">Overlays</div>
      <div class="lrow"><label><input type="checkbox" id="chkObs" checked> Satellite extent (GFM)</label></div>
      <div class="lrow"><label><input type="checkbox" id="chkDepth" checked> Forecast depth</label></div>
      <div class="lrow"><label><input type="checkbox" id="chkPop"> Population density</label></div>
      <div class="lrow"><label><input type="checkbox" id="chkMuni" checked> Municipality shading</label></div>
      <div class="lgroup">Municipality opacity</div>
      <div class="lrow" style="gap:6px;">
        <input type="range" id="muniOpSlider" min="0" max="80" value="30" style="flex:1;accent-color:#0F2044;">
        <span id="muniOpVal" style="font-size:11px;width:30px;">30%</span>
      </div>
      <button class="reset-btn" id="resetBtn">↩ Reset view</button>
    </div>`;
  L.DomEvent.disableClickPropagation(div);
  return div;
}};
lCtrl.addTo(map);

document.getElementById('lBtn').addEventListener('click',()=>document.getElementById('lPanel').classList.toggle('open'));
[['chkObs','obs'],['chkDepth','depth'],['chkPop','pop'],['chkMuni','muni']].forEach(([id,k])=>{{
  document.getElementById(id)?.addEventListener('change',function(){{layerVis[k]=this.checked;update();}});
}});
document.querySelectorAll('[name=baseMap]').forEach(r=>r.addEventListener('change',function(){{if(this.checked)setBase(this.value);}}));
document.getElementById('muniOpSlider')?.addEventListener('input',sched);
document.getElementById('resetBtn')?.addEventListener('click',()=>map.fitBounds(BOUNDS));

// ── Wire sidebar controls ─────────────────────────────────────────────────
document.getElementById('eventSelect').addEventListener('change',()=>{{
  selMuni=null; document.getElementById('muniSelect').value=''; update();
}});
document.getElementById('muniSelect').addEventListener('change',function(){{
  selMuni=this.value||null; update();
}});
document.getElementById('thrSlider').addEventListener('input',sched);

// ── Init ──────────────────────────────────────────────────────────────────
populateEvents();
populateMunis();
syncUI();
update();
</script>
</body>
</html>"""

DASHBOARD_FILENAME.write_text(html, encoding="utf-8")
print("✅ Dashboard written:", DASHBOARD_FILENAME)


In [ ]:
# ============================================================================
# STANDALONE MUNICIPALITY IMPACT EXTRACTION — Section 6 Validation
# PhilFlood / Start Ready Philippines
#
# PURPOSE
# -------
# Reads affected_population_by_adm3.csv produced by NB03 and extracts
# modelled people-affected at the three priority municipalities per basin
# across all depth thresholds and benchmark episodes.
# Compares against threshold matrix reference impact (families × HH size).
# Saves one output CSV per basin ready for figure production.
#
# HOW TO RUN
# ----------
# Option A — Standalone script:
#     python extract_muni_impact.py
#
# Option B — Jupyter cell (paste as a new cell, no NB03 kernel required):
#     Just run the cell. It finds everything from disk.
#
# CONFIGURATION
# -------------
# Set REPO_ROOT_OVERRIDE if auto-detection fails.
# Set BASIN_ID_HINT / RUN_TAG_HINT to pin to a specific run.
# ============================================================================

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# ── 0. USER OVERRIDES ─────────────────────────────────────────────────────────

REPO_ROOT_OVERRIDE = None   # e.g. Path(r"C:\pipelines\GLOFAS_ImpactFloodForecasting_PHL")
BASIN_ID_HINT      = None   # e.g. "Cagayan_01"  — None = latest run_config found
RUN_TAG_HINT       = None   # e.g. "2026-01-19_calib-test" — None = latest

AVG_HH_SIZE        = 4.0   # persons per family for reference impact conversion
SCENARIO           = "NoProt"  # filter to this scenario; None = no filter

# ── 1. Repo root (mirrors NB03 Cell 5) ────────────────────────────────────────

def _find_repo_root(start: Path = None) -> Path:
    markers = ["pyproject.toml", "setup.cfg", "setup.py", ".git", "src"]
    p = (start or Path.cwd()).resolve()
    if p.is_file():
        p = p.parent
    for parent in (p, *p.parents):
        if any((parent / m).exists() for m in markers):
            return parent
    raise RuntimeError(
        "Could not locate repo root automatically.\n"
        "Set REPO_ROOT_OVERRIDE at the top of this script."
    )

if REPO_ROOT_OVERRIDE is not None:
    REPO_ROOT = Path(REPO_ROOT_OVERRIDE).resolve()
else:
    try:
        _start = Path(__file__).parent if "__file__" in dir() else Path.cwd()
        REPO_ROOT = _find_repo_root(_start)
    except RuntimeError:
        REPO_ROOT = Path(r"C:\pipelines\GLOFAS_ImpactFloodForecasting_PHL")

DATA_PROCESSED  = REPO_ROOT / "data" / "processed"
VALIDATION_ROOT = DATA_PROCESSED / "validation"
CALIB_ROOT      = DATA_PROCESSED / "calibration" / "evt_pot"

print(f"REPO_ROOT       = {REPO_ROOT}")
print(f"DATA_PROCESSED  = {DATA_PROCESSED}")

# ── 2. Discover run_config.json (mirrors NB03 Cell 8) ─────────────────────────

def _find_run_config(calib_root: Path, basin_hint=None, tag_hint=None) -> Path:
    if basin_hint and tag_hint:
        p = calib_root / basin_hint / tag_hint / "run_config.json"
        if not p.exists():
            raise FileNotFoundError(f"run_config.json not found: {p}")
        return p
    candidates = list(calib_root.glob("*/*/run_config.json"))
    if basin_hint:
        candidates = [c for c in candidates if c.parent.parent.name == basin_hint]
    if not candidates:
        raise FileNotFoundError(
            f"No run_config.json found under: {calib_root}\n"
            "Set BASIN_ID_HINT and RUN_TAG_HINT to specify the run."
        )
    candidates.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return candidates[0]

run_config_path = _find_run_config(CALIB_ROOT, BASIN_ID_HINT, RUN_TAG_HINT)
run_config      = json.loads(run_config_path.read_text(encoding="utf-8"))

BASIN_ID = run_config.get("basin_id", "unknown")
RUN_TAG  = run_config.get("run_tag",  "unknown")

print(f"\nrun_config      = {run_config_path}")
print(f"BASIN_ID        = {BASIN_ID}")
print(f"RUN_TAG         = {RUN_TAG}")

# ── 3. Locate CSV ─────────────────────────────────────────────────────────────

OUT_DIR     = VALIDATION_ROOT / BASIN_ID / RUN_TAG
POP_OUT_DIR = OUT_DIR / "population"
CSV_PATH    = POP_OUT_DIR / "affected_population_by_adm3.csv"

if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"Population CSV not found:\n  {CSV_PATH}\n"
        "Run NB03 Cell 27 first to generate this file."
    )

df = pd.read_csv(CSV_PATH)
print(f"\nLoaded: {CSV_PATH}")
print(f"  Rows: {len(df)}  |  Columns: {list(df.columns)}")

# ── 4. Reference data from threshold matrices ──────────────────────────────────
# Source: Cagayan_RB_Flood_Threshold_Matrix.docx
#         Bicol_RB_Flood_Threshold_Matrix_Bicol_RB_Consortium.docx
# Reference population = families × AVG_HH_SIZE

PRIORITY_MUNIS = {
    "Cagayan_01": ["Enrile", "Solana", "Gattaran"],
    "Bicol_01":   ["Baao", "Buhi", "Bato"],
}

BENCHMARK_EPISODES = {
    "Cagayan_01": {
        "EP04_20201113_20201119": "Typhoon Ulysses (Vamco), Nov 2020",
        "EP08_20241119_20241119": "Typhoon Marce — peak, Nov 2024",
        "EP12_20251126_20251202": "Shearline + NE Monsoon, Nov 2025",
    },
    "Bicol_01": {
        "EP07_20241029_20241102": "STS Kristine (Trami), Oct 2024",
    },
}

REFERENCE_IMPACTS = {
    # Cagayan — per municipality, per severity level used as benchmark
    "Typhoon Ulysses (Vamco), Nov 2020": {
        "Enrile":   {"families": 7791,  "pct": 86, "severity": "VERY HIGH"},
        "Solana":   {"families": 11667, "pct": 45, "severity": "VERY HIGH"},
        "Gattaran": {"families": 1255,  "pct": 15, "severity": "HIGH"},
    },
    "Typhoon Marce — peak, Nov 2024": {
        # Matrix records Marce as VERY HIGH only for Gattaran
        "Gattaran": {"families": 5030, "pct": 50, "severity": "VERY HIGH"},
    },
    "Shearline + NE Monsoon, Nov 2025": {
        "Enrile":   {"families": 1896, "pct": 27, "severity": "MODERATE"},
        "Solana":   {"families": 161,  "pct":  3, "severity": "MODERATE"},
        "Gattaran": {"families": 418,  "pct":  5, "severity": "MODERATE"},
    },
    # Bicol — matrix reports combined total for all three municipalities only
    "STS Kristine (Trami), Oct 2024": {
        "Baao+Buhi+Bato (combined)": {
            "families": 34603, "pct": None, "severity": "VERY HIGH",
            "note": "Matrix reports combined total only — per-municipality breakdown unavailable",
        },
    },
}

# ── 5. Detect column names flexibly ───────────────────────────────────────────

def _detect_col(df, patterns):
    for pat in patterns:
        hits = [c for c in df.columns if pat.lower() in c.lower()]
        if hits:
            return hits[0]
    return None

col_ep   = _detect_col(df, ["episode_id", "episode"])
col_scen = _detect_col(df, ["scenario"])
col_thr  = _detect_col(df, ["depth_thr", "depth_threshold", "threshold"])
col_muni = _detect_col(df, ["adm3_name", "municipality", "muni_name", "adm3"])
col_mod  = _detect_col(df, ["affected_pop", "modelled_pop", "model_pop",
                              "people_affected", "pop_affected"])
col_obs  = _detect_col(df, ["observed_pop", "obs_pop", "gfm_pop"])

print(f"\nColumn mapping:")
print(f"  episode:   {col_ep}")
print(f"  scenario:  {col_scen}")
print(f"  threshold: {col_thr}")
print(f"  muni:      {col_muni}")
print(f"  modelled:  {col_mod}")
print(f"  observed:  {col_obs}  (None = not available in this file)")

missing = [(n, c) for n, c in [("episode", col_ep), ("threshold", col_thr),
                                 ("muni", col_muni), ("modelled", col_mod)]
           if c is None]
if missing:
    print(f"\nAll columns in file: {list(df.columns)}")
    raise ValueError(
        f"Could not auto-detect columns: {[n for n,_ in missing]}\n"
        "Manually assign col_ep, col_thr, col_muni, col_mod above if needed."
    )

# ── 6. Scenario filter ────────────────────────────────────────────────────────

work = df.copy()
if SCENARIO and col_scen:
    if SCENARIO in work[col_scen].values:
        work = work[work[col_scen] == SCENARIO].copy()
        print(f"\nScenario filtered to '{SCENARIO}': {len(work)} rows remaining")
    else:
        avail = work[col_scen].unique().tolist()
        print(f"\nWARNING: Scenario '{SCENARIO}' not found. Available: {avail}")
        print("Proceeding without scenario filter.")

# ── 7. Municipality matching ──────────────────────────────────────────────────

target_munis = PRIORITY_MUNIS.get(BASIN_ID, [])
target_eps   = BENCHMARK_EPISODES.get(BASIN_ID, {})

if not target_munis:
    print(f"\nWARNING: No priority municipalities defined for BASIN_ID='{BASIN_ID}'")
    print(f"Defined basins: {list(PRIORITY_MUNIS.keys())}")

avail_munis = work[col_muni].dropna().unique()
print(f"\nMunicipalities in data ({len(avail_munis)} total), first 30:")
print("  " + ", ".join(sorted(avail_munis)[:30]))

def _match_munis(targets, available):
    matched = {}
    ci = {a.lower(): a for a in available}
    for t in targets:
        if t in available:                              # exact
            matched[t] = t
        elif t.lower() in ci:                           # case-insensitive exact
            matched[t] = ci[t.lower()]
            print(f"  '{t}' matched '{matched[t]}' (case-insensitive)")
        else:
            partial = [a for a in available if t.lower() in a.lower()]
            if partial:
                matched[t] = partial[0]
                print(f"  '{t}' matched '{matched[t]}' (partial)")
            else:
                print(f"  WARNING: '{t}' — no match found. "
                      f"Check spelling against the list above.")
    return matched

muni_map = _match_munis(target_munis, avail_munis)
print(f"\nFinal municipality map: {muni_map}")

# ── 8. Episode filter ─────────────────────────────────────────────────────────

avail_eps   = set(work[col_ep].unique())
matched_eps = {ep: lbl for ep, lbl in target_eps.items() if ep in avail_eps}
missing_eps = [ep for ep in target_eps if ep not in avail_eps]

if missing_eps:
    print(f"\nWARNING: {len(missing_eps)} episode(s) not found in data:")
    for ep in missing_eps:
        print(f"  missing → '{ep}'  ({target_eps[ep]})")
    print(f"\nEpisode IDs present in data:")
    print("  " + "\n  ".join(sorted(avail_eps)))
    print("\nIf format differs (e.g. no date suffix) update BENCHMARK_EPISODES above.")

if not matched_eps:
    raise ValueError(
        "No benchmark episodes matched the data.\n"
        "Check episode ID format in BENCHMARK_EPISODES against the list printed above."
    )

print(f"\nMatched {len(matched_eps)}/{len(target_eps)} benchmark episodes.")

# ── 9. Filter and reshape ─────────────────────────────────────────────────────

filt = work[
    work[col_ep].isin(matched_eps) &
    work[col_muni].isin(muni_map.values())
].copy()

filt = filt.rename(columns={
    col_ep:   "episode_id",
    col_thr:  "depth_thr_m",
    col_muni: "municipality",
    col_mod:  "modelled_pop",
})

# Map data names back to canonical target names
inv_map = {v: k for k, v in muni_map.items()}
filt["municipality"] = filt["municipality"].map(inv_map).fillna(filt["municipality"])

if col_obs and col_obs in df.columns:
    filt["observed_pop"] = work.loc[filt.index, col_obs].values
else:
    filt["observed_pop"] = np.nan

filt["event_label"] = filt["episode_id"].map(matched_eps)
filt["depth_thr_m"] = pd.to_numeric(filt["depth_thr_m"], errors="coerce")
filt = filt.sort_values(["episode_id", "municipality", "depth_thr_m"]).reset_index(drop=True)

# ── 10. Print console summary ─────────────────────────────────────────────────

print("\n" + "=" * 80)
print("MODELLED IMPACT — PRIORITY MUNICIPALITIES × BENCHMARK EVENTS")
print("=" * 80)

for ep_id, ep_label in matched_eps.items():
    sub = filt[filt["episode_id"] == ep_id]
    print(f"\n{'─'*80}")
    print(f"  {ep_label}  ({ep_id})")
    print(f"{'─'*80}")

    if sub.empty:
        print("  No rows after filtering.")
        continue

    pivot = sub.pivot_table(
        index="depth_thr_m",
        columns="municipality",
        values="modelled_pop",
        aggfunc="sum",
    )
    print("\n  Modelled people affected by depth threshold (m):")
    print("  " + pivot.to_string(
        float_format=lambda x: f"{x:,.0f}"
    ).replace("\n", "\n  "))

    ref = REFERENCE_IMPACTS.get(ep_label, {})
    if ref:
        print(f"\n  Reference — threshold matrix (× {AVG_HH_SIZE} persons/family):")
        for muni, vals in ref.items():
            fam     = vals["families"]
            pct     = vals.get("pct")
            sev     = vals["severity"]
            ref_pop = int(round(fam * AVG_HH_SIZE))
            pct_str = f"  ({pct}% of total HHs)" if pct else ""
            note    = f"  [{vals['note']}]" if "note" in vals else ""
            print(f"    {muni}: {fam:,} families{pct_str} ≈ {ref_pop:,} persons  "
                  f"[{sev}]{note}")

# ── 11. Save CSV ──────────────────────────────────────────────────────────────

out_cols = ["episode_id", "event_label", "municipality",
            "depth_thr_m", "modelled_pop", "observed_pop"]
out_df   = filt[[c for c in out_cols if c in filt.columns]].copy()

out_path = POP_OUT_DIR / f"muni_impact_extract_{BASIN_ID}.csv"
out_df.to_csv(out_path, index=False)

print(f"\n{'='*80}")
print(f"✅  Saved: {out_path}")
print(f"    Rows: {len(out_df)}  |  Municipalities: {out_df['municipality'].nunique()}  |  "
      f"Episodes: {out_df['episode_id'].nunique()}  |  "
      f"Depth thresholds: {out_df['depth_thr_m'].nunique()}")
print(f"{'='*80}")
print(f"\nSend '{out_path.name}' for figure production.")